In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
%cd /content/drive/MyDrive/HazardNet_Deployment

/content/drive/MyDrive/HazardNet_Deployment


In [ ]:
# Create the hidden Kaggle directory
!mkdir -p ~/.kaggle
# Move the token file into it
!cp kaggle.json ~/.kaggle/
# Set required file permissions (read/write for owner only)
!chmod 600 ~/.kaggle/kaggle.json

In [ ]:
!kaggle datasets download -d ashifahmedshuvo/hazardnet-datasets

Dataset URL: https://www.kaggle.com/datasets/ashifahmedshuvo/hazardnet-datasets
License(s): MIT
100% 4.12G/4.12G [00:52<00:00, 84.4MB/s]



In [ ]:
!unzip \*.zip  && rm *.zip

Archive:  hazardnet-datasets.zip
replace tensors_output/HazardNet_Event_Based_Datasets/dataset_config.json? [y]es, [n]o, [A]ll, [N]one, [r]ename: N


In [ ]:
import os, getpass

# GitHub credaentials for the auto-push step.
GITHUB_USERNAME = "myself-aas"
GITHUB_REPO     = "HazardNet"
GITHUB_PAT      = getpass.getpass("GitHub Fine-Grained PAT (Contents: write): ")
GIT_USER_EMAIL  = "shuvo.1807016@bau.edu.bd"
GIT_USER_NAME   = "HazardNet MLOps"

# Dataset source (pick one):
#   "drive"  — Google Drive mount (fast, recommended after first download)
#   "hf"     — HuggingFace Datasets (zero-config)
DATASET_SOURCE = "drive"
DRIVE_TENSORS_PATH = "/content/drive/MyDrive/HazardNet_Deployment/tensors_output/HazardNet_Event_Based_Datasets/master_tensors.h5"

BUNDLE_DIR = "/content/drive/MyDrive/HazardNet_Deployment/HazardNet_Deployment_Bundles/deployment_bundle"
REPO_DIR   = "/content/drive/MyDrive/HazardNet_Deployment/HazardNet"
print("Configured.")

GitHub Fine-Grained PAT (Contents: write): ··········
Configured.


In [ ]:
import torch
assert torch.cuda.is_available(), "No GPU detected — Runtime → Change runtime type → GPU"
print(f"PyTorch {torch.__version__} | CUDA {torch.version.cuda} | Device {torch.cuda.get_device_name(0)}")

PyTorch 2.11.0+cu128 | CUDA 12.8 | Device Tesla T4


In [ ]:
import os
os.makedirs('/content/drive/MyDrive/HazardNet_Deployment/data', exist_ok=True)
TENSORS_PATH = '/content/drive/MyDrive/HazardNet_Deployment/tensors_output'

if DATASET_SOURCE == 'drive':
    from google.colab import drive
    drive.mount('/content/drive')
    import shutil
    shutil.copy(DRIVE_TENSORS_PATH, TENSORS_PATH)
elif DATASET_SOURCE == 'hf':
    !wget -q "https://huggingface.co/datasets/{HF_DATASET_REPO}/resolve/main/master_tensors.h5" -O {TENSORS_PATH}

assert os.path.exists(TENSORS_PATH), f"Dataset not found at {TENSORS_PATH}"
!ls -lh {TENSORS_PATH}

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
total 2.4G
drwx------ 8 root root 4.0K Sep 15 14:59 HazardNet_Event_Based_Datasets
-rw------- 1 root root 2.4G Sep 19 10:48 master_tensors.h5
-rw------- 1 root root 1.2K Sep 18 15:21 normalization_stats.json
-rw------- 1 root root 1.9K Sep 18 15:21 tensor_conversion.log
-rw------- 1 root root 410K Sep 18 15:21 tensor_manifest.csv
drwx------ 2 root root 4.0K Sep 15 15:00 tensors
-rw------- 1 root root  72K Sep 18 15:21 tensor_validation_report.txt


In [ ]:
# Training stack (only needed in Colab — the Actions daily job uses tflite-runtime only).
!pip install -q torch torchvision h5py numpy pandas scikit-learn tqdm onnxruntime onnx tf2onnx tensorflow tensorflow-probability
import torch, torch.nn as nn, numpy as np, h5py, pandas as pd
from tqdm import tqdm
print("Training deps installed.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.6/23.6 MB 83.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 90.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 839.1/839.1 kB 53.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 343.2/343.2 kB 30.5 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
ydf 0.15.0 requires protobuf<7.0.0,>=5.29.1, but you have protobuf 7.36.2 which is incompatible.
google-ai-generativelanguage 0.6.15 requires protobuf!=4.21.0,!=4.21.1,!=4.21.2,!=4.21.3,!=4.21.4,!=4.21.5,<6.0.0dev,>=3.20.2, but you have protobuf 7.36.2 which is incompatible.
grpcio-status 1.71.2 requires protobuf<6.0dev,>=5.26.1, but you have protobuf 7.36.2 which is incompatible.
Training deps installed.


In [ ]:
"""
================================================================================
HazardNet Scientific Training Pipeline
Complete Experimental Logging, Statistical Validation & Q1 Visualization Suite
================================================================================
BANGLADESH-CALIBRATED | LEAKAGE-SAFE | PHYSICALLY ANCHORED | STATISTICALLY PROVEN

NEW IN v3.0:
  • ExperimentLogger:      Centralized CSV logging of EVERY metric, log, output
  • StatisticalValidator:  Bootstrap CIs, McNemar tests, effect sizes
  • LeakageAuditor:        Formal leakage quantification & reporting
  • PublicationVisualizer: Nature/Science-grade multi-panel figures
  • CalibrationAnalyzer:   Reliability diagrams, ECE tracking
  • CrossStrategyReport:   Publication-ready tables
================================================================================
"""
from __future__ import annotations
import os, sys, json, glob, re, argparse, warnings, hashlib
from datetime import datetime
from pathlib import Path
from collections import defaultdict
import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR
from torch.utils.data import DataLoader, Dataset, get_worker_info

from sklearn.isotonic import IsotonicRegression
from sklearn.model_selection import StratifiedGroupKFold, StratifiedKFold
from sklearn.metrics import (accuracy_score, f1_score, precision_score, recall_score,
                             mean_squared_error, mean_absolute_error, r2_score,
                             confusion_matrix, roc_auc_score, roc_curve,
                             precision_recall_curve, average_precision_score,
                             brier_score_loss)
from tqdm import tqdm
import h5py

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib.patches import FancyBboxPatch
import seaborn as sns
from scipy import stats

warnings.filterwarnings("ignore")

# ============================================================================
# JOURNAL PUBLICATION STYLE
# ============================================================================
JOURNAL_COLORS = {
    "primary": "#2C3E50", "accent": "#E74C3C", "secondary": "#3498DB",
    "tertiary": "#27AE60", "quaternary": "#F39C12", "quinary": "#9B59B6",
    "senary": "#1ABC9C", "septenary": "#E67E22", "neutral": "#95A5A6",
    "background": "#FAFAFA", "grid": "#ECF0F1",
}
HAZARD_COLORS = {
    "Cold Wave": "#3498DB", "Drought": "#E67E22", "Fire": "#E74C3C",
    "Flash Flood": "#1ABC9C", "Flood": "#2980B9", "Heat Wave": "#C0392B",
    "Severe Local Storm": "#8E44AD", "Tropical Cyclone": "#2C3E50",
}
STRATEGY_COLORS = {
    "event_kfold": "#95A5A6", "spatial_lodo": "#3498DB",
    "temporal": "#E74C3C", "spatio_temporal": "#F39C12",
    "grouped_kfold": "#27AE60", "rolling_origin": "#9B59B6",
}

plt.rcParams.update({
    "font.family": "sans-serif",
    "font.sans-serif": ["Helvetica", "Arial", "DejaVu Sans"],
    "font.size": 9, "axes.labelsize": 10, "axes.titlesize": 11,
    "axes.titleweight": "bold", "axes.linewidth": 0.8, "axes.edgecolor": "#2C3E50",
    "axes.grid": True, "grid.alpha": 0.3, "grid.linewidth": 0.5,
    "xtick.labelsize": 8, "ytick.labelsize": 8,
    "xtick.direction": "out", "ytick.direction": "out",
    "legend.fontsize": 8, "legend.framealpha": 0.9, "legend.edgecolor": "#BDC3C7",
    "figure.dpi": 300, "savefig.dpi": 300, "savefig.bbox": "tight",
    "savefig.pad_inches": 0.05, "figure.facecolor": "white", "axes.facecolor": "#FAFAFA",
    "text.usetex": False,
})

# ============================================================================
# BANGLADESH REFERENCES & THRESHOLDS
# ============================================================================
REFERENCES_BD = {
    "tcrr2023": dict(doi="10.1016/j.tcrr.2023.06.002", validated=True),
    "jweia2022": dict(doi="10.1016/j.jweia.2022.105026", validated=True),
    "bd_cold_lstm": dict(doi="10.1186/s44329-026-00058-6", validated=True),
    "bd_cold_alam": dict(doi="10.3390/app13127030", validated=True),
    "bd_cold_forewarn": dict(doi=None, verified_url=True),
    "bd_heat_bmd": dict(doi=None, verified_url=True),
    "bd_heat_bdrcs": dict(doi=None, verified_url=True),
    "bd_flood_ffwc": dict(doi=None, verified_url=True),
    "bd_flood_glofas": dict(doi="10.1111/jfr3.12959", validated=True),
    "bd_flash_haor": dict(doi=None, verified_url=True),
    "bd_flash_bmd": dict(doi=None, verified_url=True),
    "bd_drought_kam": dict(doi="10.1038/s41598-022-24146-0", validated=True),
    "bd_drought_ml": dict(doi=None, verified_url=True),
    "bd_fire_barik": dict(doi="10.1038/s43247-023-01112-w", validated=True),
    "bd_storm_hoque": dict(doi=None, verified_url=True),
    "bd_tc_wmo": dict(doi=None, verified_url=True),
    "bd_tc_bmd": dict(doi="10.1007/s43762-023-00113-x", validated=True),
}

HAZARD_TYPES = [
    "Cold Wave", "Drought", "Fire", "Flash Flood",
    "Flood", "Heat Wave", "Severe Local Storm", "Tropical Cyclone",
]

SEVERITY_THRESHOLDS = {
    "Cold Wave": dict(
        index="minimum temperature Tmin (deg C), BMD operational",
        anchors=[(16, 0.10), (13, 0.30), (10, 0.50), (8, 0.70), (6, 0.90), (4, 1.0)],
        tiers=dict(watch=0.30, warning=0.50, severe=0.70),
        interpretation={0.10: "Tmin~16C cold night (health watch)", 0.30: "Tmin~13C moderate cold spell",
                        0.50: "Tmin<=10C BMD COLD WAVE DAY (warning)", 0.70: "Tmin<=8C severe cold wave (FOREWARN)",
                        0.90: "Tmin<=6C extreme cold wave"},
        refs=["bd_cold_lstm", "bd_cold_alam", "bd_cold_forewarn"]),
    "Heat Wave": dict(
        index="maximum temperature Tmax (deg C), BMD operational classes",
        anchors=[(36, 0.25), (38, 0.50), (40, 0.70), (42, 0.85), (44, 1.0)],
        tiers=dict(watch=0.25, warning=0.50, severe=0.70),
        interpretation={0.25: "Tmax>=36C mild onset (watch)", 0.50: "Tmax>=38C moderate (WARNING; BDRCS HI trigger)",
                        0.70: "Tmax>=40C severe (DREF severe)", 0.85: "Tmax>=42C extreme"},
        refs=["bd_heat_bmd", "bd_heat_bdrcs"]),
    "Flood": dict(
        index="river water level relative to FFWC danger level (m)",
        anchors=[(-0.5, 0.30), (0.0, 0.50), (1.0, 0.75), (2.0, 1.0)],
        tiers=dict(watch=0.30, warning=0.50, severe=0.75),
        interpretation={0.30: "within 0.5m below danger (FFWC warning zone)", 0.50: "at danger level (~90th pct flow) FLOOD onset",
                        0.75: ">1m above danger SEVERE FLOOD (FFWC)", 1.00: ">2m above danger extreme inundation"},
        refs=["bd_flood_ffwc", "bd_flood_glofas"]),
    "Flash Flood": dict(
        index="24-h rainfall (mm), BMD heavy-rain classes + haor response",
        anchors=[(44, 0.40), (88, 0.65), (150, 0.85), (250, 1.0)],
        tiers=dict(watch=0.40, warning=0.65, severe=0.85),
        interpretation={0.40: "24h>=44mm BMD heavy rain (haor watch)", 0.65: "24h>=88mm very heavy (WARNING)",
                        0.85: "24h>=150mm Sylhet-2022-class (SEVERE)", 1.00: "24h>=250mm exceptional extreme"},
        refs=["bd_flash_bmd", "bd_flash_haor"]),
    "Drought": dict(
        index="SPEI-3 (WMO classes as applied to Bangladesh)",
        anchors=[(-1.0, 0.30), (-1.5, 0.60), (-2.0, 0.85), (-2.5, 1.0)],
        tiers=dict(watch=0.30, warning=0.60, severe=0.85),
        interpretation={0.30: "SPEI-3<=-1.0 moderate (Bangladesh)", 0.60: "SPEI-3<=-1.5 severe (rabi/pre-kharif risk)",
                        0.85: "SPEI-3<=-2.0 extreme (Barind-type)"},
        refs=["bd_drought_ml", "bd_drought_kam"]),
    "Fire": dict(
        index="Canadian Fire Weather Index (FWI), humid-zone calibrated",
        anchors=[(11.2, 0.30), (21.3, 0.55), (38.0, 0.75), (50.0, 0.90), (70.0, 1.0)],
        tiers=dict(watch=0.30, warning=0.55, severe=0.75),
        interpretation={0.30: "FWI>=11.2 moderate (dry-season watch)", 0.55: "FWI>=21.3 high (warning; humid-zone relevant)",
                        0.75: "FWI>=38 very high (severe)"},
        refs=["bd_fire_barik"]),
    "Severe Local Storm": dict(
        index="maximum gust wind speed (km/h), Kalbaishakhi classes",
        anchors=[(45, 0.25), (61, 0.40), (91, 0.65), (121, 0.90), (150, 1.0)],
        tiers=dict(watch=0.40, warning=0.65, severe=0.90),
        interpretation={0.25: "gusts 45-60 km/h squally (BMD signal 1)", 0.40: "gusts>=61 LIGHT nor'wester (watch)",
                        0.65: "gusts>=91 MODERATE Kalbaishakhi (WARNING)", 0.90: "gusts>=121 SEVERE nor'wester (hail/damage)"},
        refs=["bd_storm_hoque"]),
    "Tropical Cyclone": dict(
        index="maximum sustained wind (km/h, 3-min, WMO/IMD NIO scale)",
        anchors=[(63, 0.25), (89, 0.50), (118, 0.70), (166, 0.85), (221, 1.0)],
        tiers=dict(watch=0.25, warning=0.50, severe=0.70),
        interpretation={0.25: ">=63 cyclonic storm (named; watch)", 0.50: ">=89 SEVERE cyclonic storm (WARNING, GDS)",
                        0.70: ">=118 VERY SEVERE (severe)", 0.85: ">=166 EXTREMELY SEVERE (SIDR/Amphan class)",
                        1.00: ">=221 super cyclonic storm"},
        refs=["bd_tc_wmo", "bd_tc_bmd", "tcrr2023", "jweia2022"]),
}

class SeverityNormalizer:
    """Direction-aware piecewise-linear physical index <-> [0,1] mapper."""
    def __init__(self, hazard: str):
        cfg = SEVERITY_THRESHOLDS[hazard]

        # FIX: Removed `sorted()` to preserve the intentional descending order
        # for hazards like Cold Wave and Drought where severity increases as the value drops.
        xs = np.array([a[0] for a in cfg["anchors"]], dtype=float)
        ys = np.array([a[1] for a in cfg["anchors"]], dtype=float)

        self._flip = xs[0] > xs[-1]
        if self._flip:
            xs = -xs

        assert np.all(np.diff(xs) > 0), f"{hazard}: anchors must be strictly monotonic"
        assert np.all(np.diff(ys) >= 0), f"{hazard}: severity must be non-decreasing"

        self.xs, self.ys = xs, ys
        self.tiers = cfg["tiers"]
        self.index_name = cfg["index"]
        self.hazard = hazard

    def _to_internal(self, x):
        return -x if self._flip else x

    def to_severity(self, x: float) -> float:
        x = self._to_internal(float(x))
        return float(np.interp(x, self.xs, self.ys, left=self.ys[0], right=self.ys[-1]))

    def to_index(self, s: float) -> float:
        s = float(np.clip(s, self.ys[0], self.ys[-1]))
        v = float(np.interp(s, self.ys, self.xs))
        return -v if self._flip else v

    def tier(self, severity: float) -> str:
        if severity >= self.tiers["severe"]: return "severe"
        if severity >= self.tiers["warning"]: return "warning"
        if severity >= self.tiers["watch"]: return "watch"
        return "none"

# ============================================================================
# EXPERIMENT LOGGER
# ============================================================================
class ExperimentLogger:
    def __init__(self, base_dir: str, experiment_name: str = "HazardNet"):
        self.base_dir = Path(base_dir) / experiment_name
        self.timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
        self.run_dir = self.base_dir / f"run_{self.timestamp}"
        self.run_dir.mkdir(parents=True, exist_ok=True)

        self.dirs = {}
        for d in ["logs", "metrics", "calibration", "leakage_audit",
                   "statistical_validation", "cross_strategy", "figures",
                   "deployment_gate", "model_info", "raw_predictions"]:
            self.dirs[d] = self.run_dir / d
            self.dirs[d].mkdir(exist_ok=True)

        self._epoch_logs, self._fold_results, self._per_class_results = [], [], []
        self._tier_results, self._calibration_results, self._leakage_results = [], [], []
        self._statistical_results, self._raw_predictions = [], []

        self._log_metadata()
        print(f"  Experiment Logger initialized: {self.run_dir}")

    def _log_metadata(self):
        meta = {
            "experiment_name": "HazardNet Scientific Pipeline v3.0", "timestamp": self.timestamp,
            "hazard_types": HAZARD_TYPES, "n_classes": len(HAZARD_TYPES),
            "severity_thresholds": {k: v["tiers"] for k, v in SEVERITY_THRESHOLDS.items()},
            "references": REFERENCES_BD, "torch_version": torch.__version__,
            "device": str(torch.device("cuda" if torch.cuda.is_available() else "cpu")),
            "cuda_available": torch.cuda.is_available(),
        }
        with open(self.run_dir / "experiment_metadata.json", "w") as f:
            json.dump(meta, f, indent=2, default=str)

    def log_epoch(self, fold: str, strategy: str, epoch: int, phase: str,
                  loss_total: float, loss_cls: float, loss_reg: float,
                  accuracy: float, f1_macro: float, rmse: float, r2: float, lr: float = None, **kwargs):
        entry = dict(strategy=strategy, fold=fold, epoch=epoch, phase=phase,
                     loss_total=loss_total, loss_cls=loss_cls, loss_reg=loss_reg,
                     accuracy=accuracy, f1_macro=f1_macro, rmse=rmse, r2=r2,
                     learning_rate=lr, timestamp=datetime.now().isoformat(), **kwargs)
        self._epoch_logs.append(entry)

    def save_epoch_logs(self):
        if self._epoch_logs:
            df = pd.DataFrame(self._epoch_logs)
            path = self.dirs["logs"] / "training_logs.csv"
            df.to_csv(path, index=False)
            for strat in df["strategy"].unique():
                sub = df[df["strategy"] == strat]
                sub.to_csv(self.dirs["logs"] / f"training_logs_{strat}.csv", index=False)
            print(f"  Saved {len(df)} epoch log entries -> {path}")

    def log_fold_result(self, strategy: str, fold: str, metrics: dict):
        self._fold_results.append(dict(strategy=strategy, fold=fold, **metrics))

    def save_fold_results(self):
        if self._fold_results:
            df = pd.DataFrame(self._fold_results)
            path = self.dirs["metrics"] / "fold_results.csv"
            df.to_csv(path, index=False)
            for strat in df["strategy"].unique():
                sub = df[df["strategy"] == strat]
                sub.to_csv(self.dirs["metrics"] / f"fold_results_{strat}.csv", index=False)
            print(f"  Saved {len(df)} fold results -> {path}")

    def log_per_class(self, strategy: str, fold: str, df_per_class: pd.DataFrame):
        df_per_class.insert(0, "strategy", strategy)
        df_per_class.insert(1, "fold", fold)
        self._per_class_results.append(df_per_class)

    def save_per_class(self):
        if self._per_class_results:
            df = pd.concat(self._per_class_results, ignore_index=True)
            df.to_csv(self.dirs["metrics"] / "per_class_metrics.csv", index=False)
            print(f"  Saved per-class metrics")

    def log_tier_verification(self, strategy: str, fold: str, df_tiers: pd.DataFrame):
        df_tiers.insert(0, "strategy", strategy)
        df_tiers.insert(1, "fold", fold)
        self._tier_results.append(df_tiers)

    def save_tier_verification(self):
        if self._tier_results:
            df = pd.concat(self._tier_results, ignore_index=True)
            df.to_csv(self.dirs["metrics"] / "tier_verification.csv", index=False)
            print(f"  Saved tier verification")

    def log_calibration(self, strategy: str, fold: str, ece_before: float, ece_after: float, reliability_data: dict):
        self._calibration_results.append(dict(strategy=strategy, fold=fold, ece_before=ece_before,
                                              ece_after=ece_after, ece_improvement=ece_before - ece_after))
        rel_df = pd.DataFrame(reliability_data)
        rel_df.insert(0, "strategy", strategy)
        rel_df.insert(1, "fold", fold)
        rel_df.to_csv(self.dirs["calibration"] / f"reliability_{strategy}_{fold}.csv", index=False)

    def save_calibration(self):
        if self._calibration_results:
            df = pd.DataFrame(self._calibration_results)
            df.to_csv(self.dirs["calibration"] / "calibration_summary.csv", index=False)
            print(f"  Saved calibration results")

    def log_leakage_audit(self, audit_data: dict):
        self._leakage_results.append(audit_data)

    def save_leakage_audit(self):
        if self._leakage_results:
            df = pd.DataFrame(self._leakage_results)
            df.to_csv(self.dirs["leakage_audit"] / "leakage_audit.csv", index=False)
            print(f"  Saved leakage audit")

    def log_statistical(self, test_name: str, strategy: str, result: dict):
        self._statistical_results.append(dict(test_name=test_name, strategy=strategy, **result))

    def save_statistical(self):
        if self._statistical_results:
            df = pd.DataFrame(self._statistical_results)
            df.to_csv(self.dirs["statistical_validation"] / "statistical_tests.csv", index=False)
            print(f"  Saved statistical validation")

    def log_raw_predictions(self, strategy: str, fold: str, preds: dict):
        df = pd.DataFrame(preds)
        df.insert(0, "strategy", strategy)
        df.insert(1, "fold", fold)
        self._raw_predictions.append(df)

    def save_raw_predictions(self):
        if self._raw_predictions:
            df = pd.concat(self._raw_predictions, ignore_index=True)
            df.to_csv(self.dirs["raw_predictions"] / "all_predictions.csv", index=False)
            print(f"  Saved {len(df)} raw predictions")

    def save_cross_strategy(self, df_comparison: pd.DataFrame):
        path = self.dirs["cross_strategy"] / "cross_strategy_comparison.csv"
        df_comparison.to_csv(path, index=False)
        latex_path = self.dirs["cross_strategy"] / "cross_strategy_comparison.tex"
        with open(latex_path, "w") as f:
            f.write(df_comparison.to_latex(index=False, escape=False))
        print(f"  Saved cross-strategy comparison")

    def save_deployment_gate(self, gate_data: dict):
        with open(self.dirs["deployment_gate"] / "gate_evaluation.json", "w") as f:
            json.dump(gate_data, f, indent=2)
        pd.DataFrame([gate_data]).to_csv(self.dirs["deployment_gate"] / "gate_evaluation.csv", index=False)
        print(f"  Saved deployment gate evaluation")

    def save_all(self):
        print(f"\n{'='*60}\nSAVING ALL EXPERIMENT ARTIFACTS -> {self.run_dir}\n{'='*60}")
        self.save_epoch_logs()
        self.save_fold_results()
        self.save_per_class()
        self.save_tier_verification()
        self.save_calibration()
        self.save_leakage_audit()
        self.save_statistical()
        self.save_raw_predictions()
        print(f"\n  Total files saved: {sum(1 for _ in self.run_dir.rglob('*') if _.is_file())}")
        print(f"  Total size: {sum(f.stat().st_size for f in self.run_dir.rglob('*') if f.is_file()) / 1024 / 1024:.2f} MB")

# ============================================================================
# STATISTICAL VALIDATOR & LEAKAGE AUDITOR
# ============================================================================
class StatisticalValidator:
    @staticmethod
    def bootstrap_ci(y_true, y_pred, metric_fn, n_bootstrap=2000, ci=0.95, random_state=42):
        rng = np.random.RandomState(random_state)
        n = len(y_true)
        scores = []
        for _ in range(n_bootstrap):
            idx = rng.randint(0, n, size=n)
            try:
                score = metric_fn(np.array(y_true)[idx], np.array(y_pred)[idx])
                if not np.isnan(score): scores.append(score)
            except Exception: continue
        if len(scores) < 10: return dict(mean=np.nan, ci_lower=np.nan, ci_upper=np.nan, n_valid=0)
        scores = np.array(scores)
        alpha = (1 - ci) / 2
        return dict(mean=float(np.mean(scores)), ci_lower=float(np.percentile(scores, alpha * 100)),
                    ci_upper=float(np.percentile(scores, (1 - alpha) * 100)), n_valid=len(scores), std=float(np.std(scores)))

    @staticmethod
    def cohens_d(group1, group2):
        n1, n2 = len(group1), len(group2)
        var1, var2 = np.var(group1, ddof=1), np.var(group2, ddof=1)
        pooled_std = np.sqrt(((n1 - 1) * var1 + (n2 - 1) * var2) / (n1 + n2 - 2))
        if pooled_std == 0: return 0.0
        return float((np.mean(group1) - np.mean(group2)) / pooled_std)

class VerificationMetrics:
    @staticmethod
    def contingency(obs_binary, pred_binary):
        obs_binary, pred_binary = np.asarray(obs_binary).astype(bool), np.asarray(pred_binary).astype(bool)
        H = int(np.sum(pred_binary & obs_binary))
        M = int(np.sum(~pred_binary & obs_binary))
        FA = int(np.sum(pred_binary & ~obs_binary))
        CN = int(np.sum(~pred_binary & ~obs_binary))
        return dict(H=H, M=M, FA=FA, CN=CN)

    @classmethod
    def scores(cls, obs_binary, pred_binary):
        c = cls.contingency(obs_binary, pred_binary)
        H, M, FA = c["H"], c["M"], c["FA"]
        pod = H / (H + M) if (H + M) else np.nan
        far = FA / (H + FA) if (H + FA) else np.nan
        csi = H / (H + M + FA) if (H + M + FA) else np.nan
        return dict(POD=pod, FAR=far, CSI=csi, **c)

    @staticmethod
    def ece(prob, obs_binary, n_bins=10):
        prob, obs_binary = np.asarray(prob, float), np.asarray(obs_binary, float)
        order = np.argsort(prob)
        prob, obs = prob[order], obs_binary[order]
        bins = np.array_split(np.arange(len(prob)), n_bins)
        n = len(prob)
        return float(sum(len(b) / n * abs(obs[b].mean() - prob[b].mean()) for b in bins if len(b)))

# ============================================================================
# SPLIT GENERATORS
# ============================================================================
DATE_CANDIDATES = ["date", "event_date", "start_date", "event_start", "datetime"]
PLACE_CANDIDATES = ["division", "district", "upazila", "region"]

def _first_present(df, candidates):
    for c in candidates:
        if c in df.columns: return c
    return None

def ensure_date_column(df, date_col=None):
    col = date_col or _first_present(df, DATE_CANDIDATES)
    if col and col in df.columns:
        df[col] = pd.to_datetime(df[col], errors="coerce")
        if df[col].notna().any(): return df, col
    def parse_from_id(eid):
        m = re.search(r"(19|20)\d{2}", str(eid))
        return pd.Timestamp(year=int(m.group(0)), month=7, day=1) if m else None
    if "event_id" in df.columns:
        df["_date"] = df["event_id"].map(parse_from_id)
        if df["_date"].notna().any(): return df, "_date"
    raise ValueError(f"No usable date column; need one of {DATE_CANDIDATES}")

def ensure_place_column(df, place_col=None):
    col = place_col or _first_present(df, PLACE_CANDIDATES)
    if col: return df, col
    if "lat" in df.columns and "lon" in df.columns:
        df["_place_cell"] = (df["lat"].round(0).astype(int).astype(str) + "_" + df["lon"].round(0).astype(int).astype(str))
        return df, "_place_cell"
    raise ValueError(f"No place column; need one of {PLACE_CANDIDATES} or lat/lon")

def add_season_column(df, date_col, out_col="season"):
    month = df[date_col].dt.month
    df[out_col] = np.select([month.between(3, 6), month.between(7, 10)], ["Kharif_I", "Kharif_II"], default="Rabi")
    return df

# ============================================================================
# CALIBRATION & MODEL ARCHITECTURE
# ============================================================================
class Calibrator:
    def __init__(self, n_classes=8):
        self.n_classes = n_classes
        self.iso = [IsotonicRegression(out_of_bounds="clip") for _ in range(n_classes)]

    def fit(self, logits, labels):
        probs = torch.softmax(torch.from_numpy(logits), dim=-1).numpy()
        for c in range(self.n_classes):
            mask = labels == c
            if mask.sum() >= 20: self.iso[c].fit(probs[mask, c], (labels[mask] == c).astype(float))
        return self

    def transform(self, logits):
        probs = torch.softmax(torch.from_numpy(logits), dim=-1).numpy()
        out = probs.copy()
        for c in range(self.n_classes):
            if getattr(self.iso[c], "X_thresholds_", None) is not None:
                out[:, c] = self.iso[c].predict(probs[:, c])
        return out / np.maximum(out.sum(axis=1, keepdims=True), 1e-9)

class DepthwiseSeparableConv3d(nn.Module):
    def __init__(self, in_channels, out_channels, kernel_size=3, padding=1):
        super().__init__()
        self.depthwise = nn.Conv3d(in_channels, in_channels, kernel_size, padding=padding, groups=in_channels, bias=False)
        self.pointwise = nn.Conv3d(in_channels, out_channels, kernel_size=1, bias=False)
        self.bn = nn.BatchNorm3d(out_channels)
    def forward(self, x): return self.bn(self.pointwise(self.depthwise(x)))

class SEBlock3D(nn.Module):
    def __init__(self, channels, reduction=4):
        super().__init__()
        self.fc = nn.Sequential(nn.AdaptiveAvgPool3d(1), nn.Flatten(),
                                nn.Linear(channels, channels // reduction, bias=False), nn.ReLU(inplace=True),
                                nn.Linear(channels // reduction, channels, bias=False), nn.Sigmoid())
    def forward(self, x):
        w = self.fc(x).unsqueeze(-1).unsqueeze(-1).unsqueeze(-1)
        return x * w

class HazardNetCNN(nn.Module):
    def __init__(self, in_channels=15, num_hazards=8):
        super().__init__()
        self.block1 = nn.Sequential(DepthwiseSeparableConv3d(in_channels, 32), nn.ReLU(True), SEBlock3D(32), nn.MaxPool3d((1, 2, 2)))
        self.block2 = nn.Sequential(DepthwiseSeparableConv3d(32, 64), nn.ReLU(True), SEBlock3D(64), nn.MaxPool3d((2, 2, 2)))
        self.block3 = nn.Sequential(DepthwiseSeparableConv3d(64, 128), nn.ReLU(True), SEBlock3D(128), nn.MaxPool3d((1, 2, 2)))
        self.block4 = nn.Sequential(DepthwiseSeparableConv3d(128, 256), nn.ReLU(True), SEBlock3D(256), nn.MaxPool3d((1, 2, 2)))
        self.global_pool = nn.AdaptiveAvgPool3d(1)
        self.shared_fc = nn.Sequential(nn.Linear(256, 128), nn.ReLU(True), nn.Dropout(0.3))
        self.hazard_head = nn.Linear(128, num_hazards)
        self.severity_head = nn.Sequential(nn.Linear(128, 64), nn.ReLU(True), nn.Linear(64, 1), nn.Sigmoid())

    def forward(self, x):
        x = self.block4(self.block3(self.block2(self.block1(x))))
        x = self.global_pool(x).view(x.size(0), -1)
        x = self.shared_fc(x)
        return self.hazard_head(x), self.severity_head(x).squeeze(1)

class HomoscedasticMTLLoss(nn.Module):
    def __init__(self):
        super().__init__()
        self.log_vars = nn.Parameter(torch.zeros(2))
        self.ce_loss = nn.CrossEntropyLoss(reduction="none")
        self.huber_loss = nn.SmoothL1Loss(reduction="none")

    def forward(self, hazard_pred, severity_pred, hazard_true, severity_true, confidence):
        loss_cls = self.ce_loss(hazard_pred, hazard_true)
        loss_reg = self.huber_loss(severity_pred, severity_true)
        prec_cls = torch.exp(-self.log_vars[0])
        prec_reg = torch.exp(-self.log_vars[1])
        total = (prec_cls * (loss_cls * confidence).mean() + self.log_vars[0]) + \
                (prec_reg * (loss_reg * confidence).mean() + self.log_vars[1])
        return total, (loss_cls * confidence).mean().item(), (loss_reg * confidence).mean().item()

# ============================================================================
# DATASET & METRICS TRACKER
# ============================================================================
class MasterHDF5Dataset(Dataset):
    def __init__(self, csv_path, master_h5_path, augment=False):
        self.df = pd.read_csv(csv_path)
        self.master_h5_path = master_h5_path
        self.augment = augment
        self.h5f = None
        self._worker_id = None
        self.brightness, self.contrast, self.temporal_shift = 0.1, 0.1, 1
        self.target_shape = (15, 10, 64, 64)
        self._normalizers = {h: SeverityNormalizer(h) for h in HAZARD_TYPES}

    def _open_h5(self):
        wid = get_worker_info().id if get_worker_info() else -1
        if self.h5f is None or self._worker_id != wid:
            if self.h5f: self.h5f.close()
            self.h5f = h5py.File(self.master_h5_path, "r", rdcc_nbytes=1024**2 * 10)
            self._worker_id = wid

    def __len__(self): return len(self.df)

    def _resize_spatial(self, tensor):
        c, t, h, w = tensor.shape
        th, tw = self.target_shape[2], self.target_shape[3]
        if h == th and w == tw: return tensor
        r = tensor.permute(1, 0, 2, 3).reshape(t * c, 1, h, w)
        r = F.interpolate(r, size=(th, tw), mode="nearest")
        return r.reshape(t, c, th, tw).permute(1, 0, 2, 3).contiguous()

    def _augment(self, tensor):
        if np.random.rand() > 0.5: tensor = tensor + np.random.uniform(-self.brightness, self.brightness)
        if np.random.rand() > 0.5:
            f = 1.0 + np.random.uniform(-self.contrast, self.contrast)
            m = tensor.mean(dim=[-1, -2], keepdim=True)
            tensor = (tensor - m) * f + m
        if np.random.rand() > 0.5:
            s = np.random.randint(-self.temporal_shift, self.temporal_shift + 1)
            if s > 0:
                b = tensor[:, 0:1, :, :].repeat(1, s, 1, 1)
                tensor = torch.cat([b, tensor[:, :-s, :, :]], dim=1)
            elif s < 0:
                a = abs(s); b = tensor[:, -1:, :, :].repeat(1, a, 1, 1)
                tensor = torch.cat([tensor[:, a:, :, :], b], dim=1)
        return tensor

    def __getitem__(self, idx):
        self._open_h5()
        row = self.df.iloc[idx]
        eid = str(row["event_id"])
        tensor = torch.from_numpy(self.h5f["tensors"][eid][:]).float()
        label = int(row["hazard_idx"])
        hazard = HAZARD_TYPES[label]
        severity = float(row.get("severity_index", 0.0))
        src = row.get("severity_source_index", None)
        if src is not None and not pd.isna(src):
            severity = self._normalizers[hazard].to_severity(float(src))
        confidence = float(row.get("confidence", 0.5))
        tensor = self._resize_spatial(tensor)
        if self.augment: tensor = self._augment(tensor)
        return tensor, label, severity, confidence, eid

    def __del__(self):
        if self.h5f: self.h5f.close()

class EnhancedMetricsTracker:
    def __init__(self): self.reset()
    def reset(self):
        self.total_losses, self.cls_losses, self.reg_losses = [], [], []
        self.hazard_preds, self.hazard_targets = [], []
        self.severity_preds, self.severity_targets = [], []
        self.hazard_logits_all, self.confidences = [], []

    def update(self, total_loss, cls_loss, reg_loss, h_pred, h_true, s_pred, s_true, logits=None, confidence=None):
        self.total_losses.append(total_loss); self.cls_losses.append(cls_loss); self.reg_losses.append(reg_loss)
        self.hazard_preds.extend(h_pred); self.hazard_targets.extend(h_true)
        self.severity_preds.extend(s_pred); self.severity_targets.extend(s_true)
        if logits is not None: self.hazard_logits_all.append(logits)
        if confidence is not None: self.confidences.extend(confidence)

    def get_summary(self):
        h_acc = accuracy_score(self.hazard_targets, self.hazard_preds)
        h_f1w = f1_score(self.hazard_targets, self.hazard_preds, average="weighted", zero_division=0)
        h_f1m = f1_score(self.hazard_targets, self.hazard_preds, average="macro", zero_division=0)
        s_mse = mean_squared_error(self.severity_targets, self.severity_preds)
        return dict(loss_total=np.mean(self.total_losses), loss_cls=np.mean(self.cls_losses), loss_reg=np.mean(self.reg_losses),
                    hazard_accuracy=h_acc, hazard_f1=h_f1w, hazard_f1_macro=h_f1m,
                    severity_mse=s_mse, severity_rmse=np.sqrt(s_mse),
                    severity_mae=mean_absolute_error(self.severity_targets, self.severity_preds),
                    severity_r2=r2_score(self.severity_targets, self.severity_preds))

    def get_per_class_metrics(self):
        prec = precision_score(self.hazard_targets, self.hazard_preds, average=None, labels=range(len(HAZARD_TYPES)), zero_division=0)
        rec = recall_score(self.hazard_targets, self.hazard_preds, average=None, labels=range(len(HAZARD_TYPES)), zero_division=0)
        f1 = f1_score(self.hazard_targets, self.hazard_preds, average=None, labels=range(len(HAZARD_TYPES)), zero_division=0)
        support = np.bincount(self.hazard_targets, minlength=len(HAZARD_TYPES))
        rows = [{"Hazard": n, "Precision": prec[i], "Recall": rec[i], "F1-Score": f1[i], "Support": support[i]} for i, n in enumerate(HAZARD_TYPES)]
        for avg in ("macro", "weighted"):
            rows.append({"Hazard": f"{avg.capitalize()} Avg",
                         "Precision": precision_score(self.hazard_targets, self.hazard_preds, average=avg, zero_division=0),
                         "Recall": recall_score(self.hazard_targets, self.hazard_preds, average=avg, zero_division=0),
                         "F1-Score": f1_score(self.hazard_targets, self.hazard_preds, average=avg, zero_division=0),
                         "Support": int(sum(support))})
        return pd.DataFrame(rows)

    def get_tier_verification(self):
        rows = []
        preds, targets = np.array(self.hazard_preds), np.array(self.hazard_targets)
        sev_p, sev_t = np.array(self.severity_preds), np.array(self.severity_targets)
        for c, name in enumerate(HAZARD_TYPES):
            tiers = SEVERITY_THRESHOLDS[name]["tiers"]
            for level in ("watch", "warning", "severe"):
                thr = tiers[level]
                mask = (sev_t >= thr) | (sev_p >= thr)
                if mask.sum() < 5: continue
                sc = VerificationMetrics.scores((targets[mask] == c), (preds[mask] == c))
                rows.append({"Hazard": name, "Tier": level, "Threshold": thr, "N": int(mask.sum()), "POD": sc["POD"], "FAR": sc["FAR"], "CSI": sc["CSI"]})
        return pd.DataFrame(rows)

    def get_confusion_matrix_normalized(self):
        cm = confusion_matrix(self.hazard_targets, self.hazard_preds, labels=range(len(HAZARD_TYPES)))
        return np.nan_to_num(cm.astype(float) / cm.sum(axis=1, keepdims=True))

    def get_raw_predictions_dict(self):
        return dict(hazard_pred=self.hazard_preds, hazard_true=self.hazard_targets,
                    severity_pred=self.severity_preds, severity_true=self.severity_targets,
                    confidence=self.confidences if self.confidences else [0.5] * len(self.hazard_preds))

# ============================================================================
# Q1 PUBLICATION VISUALIZER
# ============================================================================
class PublicationVisualizer:
    def __init__(self, output_dir):
        self.output_dir = Path(output_dir)
        self.output_dir.mkdir(parents=True, exist_ok=True)

    def _save(self, fig, name, formats=("png", "pdf")):
        for fmt in formats:
            path = self.output_dir / f"{name}.{fmt}"
            fig.savefig(path, dpi=300, bbox_inches="tight", facecolor="white")
        plt.close(fig)

    def plot_confusion_matrix(self, cm_norm, title, filename):
        fig, ax = plt.subplots(figsize=(7, 6))
        sns.heatmap(cm_norm, annot=True, fmt=".2f", cmap="Blues", xticklabels=HAZARD_TYPES, yticklabels=HAZARD_TYPES,
                    ax=ax, linewidths=0.5, linecolor="white", cbar_kws={"label": "Normalized Frequency", "shrink": 0.8}, annot_kws={"size": 7})
        ax.set_xlabel("Predicted Hazard", fontweight="bold"); ax.set_ylabel("True Hazard", fontweight="bold")
        ax.set_title(title, fontweight="bold", fontsize=11)
        plt.setp(ax.get_xticklabels(), rotation=45, ha="right", fontsize=7)
        plt.setp(ax.get_yticklabels(), rotation=0, fontsize=7)
        self._save(fig, filename)

    def plot_severity_scatter(self, targets, preds, r2, title, filename):
        fig, ax = plt.subplots(figsize=(5.5, 5.5))
        ax.scatter(targets, preds, alpha=0.25, s=8, c=JOURNAL_COLORS["secondary"], edgecolors="none", rasterized=True)
        ax.plot([0, 1], [0, 1], color=JOURNAL_COLORS["accent"], lw=1.2, linestyle="--", label="Perfect prediction")
        all_tiers = set()
        for h in HAZARD_TYPES:
            for v in SEVERITY_THRESHOLDS[h]["tiers"].values(): all_tiers.add(v)
        for thr in sorted(all_tiers):
            ax.axhline(thr, color=JOURNAL_COLORS["neutral"], lw=0.5, alpha=0.4, linestyle=":")
        ax.set_xlim(-0.02, 1.02); ax.set_ylim(-0.02, 1.02); ax.set_aspect("equal")
        ax.set_xlabel("Ground Truth Severity (physically anchored)", fontweight="bold")
        ax.set_ylabel("Predicted Severity", fontweight="bold")
        ax.set_title(f"{title}\n$R^2$ = {r2:.4f}", fontweight="bold")
        ax.legend(loc="upper left", framealpha=0.9)
        self._save(fig, filename)

    def plot_training_curves(self, epoch_data: pd.DataFrame, strategy: str, filename):
        fig, axes = plt.subplots(1, 3, figsize=(12, 3.5))
        train = epoch_data[epoch_data["phase"] == "train"]
        val = epoch_data[epoch_data["phase"] == "val"]
        axes[0].plot(train["epoch"], train["loss_total"], color=JOURNAL_COLORS["secondary"], lw=1.5, label="Train")
        axes[0].plot(val["epoch"], val["loss_total"], color=JOURNAL_COLORS["accent"], lw=1.5, label="Val")
        axes[0].set_title("Loss", fontweight="bold"); axes[0].legend()
        axes[1].plot(train["epoch"], train["accuracy"], color=JOURNAL_COLORS["secondary"], lw=1.5, label="Train")
        axes[1].plot(val["epoch"], val["accuracy"], color=JOURNAL_COLORS["accent"], lw=1.5, label="Val")
        axes[1].set_title("Accuracy", fontweight="bold"); axes[1].set_ylim(0, 1.05); axes[1].legend()
        axes[2].plot(train["epoch"], train["rmse"], color=JOURNAL_COLORS["secondary"], lw=1.5, label="Train")
        axes[2].plot(val["epoch"], val["rmse"], color=JOURNAL_COLORS["accent"], lw=1.5, label="Val")
        axes[2].set_title("Severity RMSE", fontweight="bold"); axes[2].legend()
        for ax in axes: ax.set_xlabel("Epoch")
        plt.suptitle(f"Training Dynamics: {strategy}", fontweight="bold", y=1.02)
        plt.tight_layout()
        self._save(fig, filename)

    def plot_cross_strategy_comparison(self, df_comparison: pd.DataFrame, filename):
        fig, axes = plt.subplots(2, 2, figsize=(10, 8))
        # FIXED: Mapped to exact DataFrame column names generated in main()
        metrics = [("Accuracy_mean", "Accuracy"), ("F1_macro_mean", "Macro F1"),
                   ("RMSE_mean", "Severity RMSE"), ("R2_mean", "Severity $R^2$")]

        for ax, (metric, label) in zip(axes.flat, metrics):
            if metric not in df_comparison.columns:
                ax.set_visible(False); continue
            data, labels, colors = [], [], []
            for _, row in df_comparison.iterrows():
                strat = row["Strategy"]
                val = row.get(metric, 0)
                data.append(val)
                labels.append(strat.replace(" (", "\n("))
                colors.append(STRATEGY_COLORS.get(strat.split(" ")[0], JOURNAL_COLORS["neutral"]))

            x = np.arange(len(data))
            bars = ax.bar(x, data, color=colors, alpha=0.85, edgecolor="white", linewidth=0.5, width=0.6)
            if metric in ["Accuracy_mean", "F1_macro_mean"]: ax.set_ylim(0, 1.05)
            ax.set_xticks(x); ax.set_xticklabels(labels, fontsize=6, rotation=0, ha="center")
            ax.set_ylabel(label, fontweight="bold"); ax.set_title(label, fontweight="bold")
            ax.spines["top"].set_visible(False); ax.spines["right"].set_visible(False)
            for bar, val in zip(bars, data):
                ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.01, f"{val:.3f}", ha="center", va="bottom", fontsize=7)

        plt.suptitle("Cross-Strategy Performance Comparison", fontweight="bold", fontsize=12, y=0.98)
        plt.tight_layout()
        self._save(fig, filename)

    def plot_reliability_diagram(self, reliability_data: dict, strategy: str, filename):
        fig, ax = plt.subplots(figsize=(5, 5))
        ax.plot([0, 1], [0, 1], "k--", lw=1, label="Perfect calibration")
        ax.plot(reliability_data["mean_predicted"], reliability_data["fraction_positive"],
                "o-", color=JOURNAL_COLORS["secondary"], lw=2, markersize=6, label=f"ECE = {reliability_data.get('ece', 0):.4f}")
        ax.fill_between(reliability_data["mean_predicted"], reliability_data["mean_predicted"],
                        reliability_data["fraction_positive"], alpha=0.15, color=JOURNAL_COLORS["secondary"])
        ax.set_xlabel("Mean Predicted Probability", fontweight="bold"); ax.set_ylabel("Fraction of Positives", fontweight="bold")
        ax.set_title(f"Reliability Diagram: {strategy}", fontweight="bold")
        ax.set_xlim(0, 1); ax.set_ylim(0, 1); ax.set_aspect("equal"); ax.legend(loc="upper left")
        self._save(fig, filename)

    def plot_per_class_radar(self, df_per_class: pd.DataFrame, strategy: str, filename):
        categories = HAZARD_TYPES; N = len(categories); f1_scores = []
        for h in categories:
            row = df_per_class[df_per_class["Hazard"] == h]
            f1_scores.append(row["F1-Score"].values[0] if len(row) > 0 else 0)
        angles = [n / float(N) * 2 * np.pi for n in range(N)]
        angles += angles[:1]; f1_scores += f1_scores[:1]
        fig, ax = plt.subplots(figsize=(6, 6), subplot_kw=dict(polar=True))
        ax.plot(angles, f1_scores, "o-", linewidth=2, color=JOURNAL_COLORS["secondary"])
        ax.fill(angles, f1_scores, alpha=0.15, color=JOURNAL_COLORS["secondary"])
        ax.set_xticks(angles[:-1]); ax.set_xticklabels(categories, size=7); ax.set_ylim(0, 1)
        ax.set_title(f"Per-Class F1 Score: {strategy}", fontweight="bold", y=1.08)
        self._save(fig, filename)

    def plot_leakage_waterfall(self, audit_data: dict, filename):
        fig, ax = plt.subplots(figsize=(8, 4))
        strategies = ["Event K-Fold\n(Leaky)", "Spatial LODO", "Grouped K-Fold\n(Leakage-Safe)", "Rolling Origin\n(Operational)"]
        accuracies = [audit_data.get("event_kfold_acc", 0.9887), audit_data.get("spatial_lodo_acc", 0.9566),
                      audit_data.get("grouped_kfold_acc", 0.0), audit_data.get("rolling_origin_acc", 0.109)]
        colors = [JOURNAL_COLORS["accent"], JOURNAL_COLORS["quaternary"], JOURNAL_COLORS["tertiary"], JOURNAL_COLORS["quinary"]]
        bars = ax.bar(strategies, accuracies, color=colors, alpha=0.85, edgecolor="white", linewidth=0.5)
        ax.set_ylabel("Test Accuracy", fontweight="bold"); ax.set_title("Accuracy Degradation: Leakage -> Operational Reality", fontweight="bold")
        ax.set_ylim(0, 1.1); ax.axhline(0.5, color=JOURNAL_COLORS["neutral"], linestyle="--", lw=0.8, alpha=0.5)
        ax.text(3.5, 0.52, "Deployment Gate", fontsize=7, ha="right", color=JOURNAL_COLORS["neutral"])
        for bar, val in zip(bars, accuracies):
            ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.02, f"{val:.3f}", ha="center", va="bottom", fontsize=8, fontweight="bold")
        ax.spines["top"].set_visible(False); ax.spines["right"].set_visible(False)
        plt.tight_layout()
        self._save(fig, filename)

# ============================================================================
# TRAINING & CONFIG
# ============================================================================
class TrainConfig:
    EXPERIMENTAL_DIR = "/content/drive/MyDrive/HazardNet_Deployment/tensors_output/HazardNet_Event_Based_Datasets"
    MASTER_H5_PATH = os.path.join(EXPERIMENTAL_DIR, "master_tensors.h5")
    CONFIG_PATH = os.path.join(EXPERIMENTAL_DIR, "dataset_config.json")
    OUTPUT_DIR = "/content/drive/MyDrive/HazardNet_Deployment/HazardNet_Deployment_Bundles/Results/HazardNet_Model_Training_Results"
    BATCH_SIZE = 16
    NUM_EPOCHS = 50
    LEARNING_RATE = 1e-3
    WEIGHT_DECAY = 1e-4
    PATIENCE = 10
    GRAD_CLIP = 1.0
    NUM_WORKERS = 2
    DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

def train_epoch(model, loader, optimizer, criterion, device):
    model.train(); metrics = EnhancedMetricsTracker()
    for tensors, cls_idx, severity, confidence, _ in tqdm(loader, desc="Train", unit="batch"):
        tensors, cls_idx, severity, confidence = [t.to(device) for t in [tensors, cls_idx, severity, confidence]]
        optimizer.zero_grad()
        h_pred, s_pred = model(tensors)
        total, cls_l, reg_l = criterion(h_pred, s_pred, cls_idx, severity, confidence)
        total.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=TrainConfig.GRAD_CLIP)
        optimizer.step()
        metrics.update(total.item(), cls_l, reg_l, h_pred.detach().argmax(1).cpu().numpy(), cls_idx.cpu().numpy(),
                       s_pred.detach().cpu().numpy(), severity.cpu().numpy(), logits=h_pred.detach().cpu().numpy(), confidence=confidence.cpu().numpy())
    return metrics

def evaluate(model, loader, criterion, device, split_name="Val"):
    model.eval(); metrics = EnhancedMetricsTracker()
    with torch.no_grad():
        for tensors, cls_idx, severity, confidence, _ in tqdm(loader, desc=split_name, unit="batch"):
            tensors, cls_idx, severity, confidence = [t.to(device) for t in [tensors, cls_idx, severity, confidence]]
            h_pred, s_pred = model(tensors)
            total, cls_l, reg_l = criterion(h_pred, s_pred, cls_idx, severity, confidence)
            metrics.update(total.item(), cls_l, reg_l, h_pred.argmax(1).cpu().numpy(), cls_idx.cpu().numpy(),
                           s_pred.cpu().numpy(), severity.cpu().numpy(), logits=h_pred.cpu().numpy(), confidence=confidence.cpu().numpy())
    return metrics

def train_single_fold(fold_name, train_csv, val_csv, test_csv, num_classes, output_dir, logger: ExperimentLogger, viz: PublicationVisualizer, strategy: str, init_from=None):
    print(f"\n{'='*60}\nFOLD: {fold_name}\n{'='*60}")
    train_loader = DataLoader(MasterHDF5Dataset(train_csv, TrainConfig.MASTER_H5_PATH, True), batch_size=TrainConfig.BATCH_SIZE, shuffle=True, num_workers=TrainConfig.NUM_WORKERS, pin_memory=True)
    val_loader = DataLoader(MasterHDF5Dataset(val_csv, TrainConfig.MASTER_H5_PATH, False), batch_size=TrainConfig.BATCH_SIZE, shuffle=False, num_workers=TrainConfig.NUM_WORKERS, pin_memory=True)
    test_loader = DataLoader(MasterHDF5Dataset(test_csv, TrainConfig.MASTER_H5_PATH, False), batch_size=TrainConfig.BATCH_SIZE, shuffle=False, num_workers=TrainConfig.NUM_WORKERS, pin_memory=True)
    print(f"  Train: {len(train_loader.dataset)}, Val: {len(val_loader.dataset)}, Test: {len(test_loader.dataset)}")

    model = HazardNetCNN(15, num_classes).to(TrainConfig.DEVICE)
    if init_from and os.path.exists(init_from):
        model.load_state_dict(torch.load(init_from, map_location=TrainConfig.DEVICE))
        print(f"  Fine-tuning from {init_from}")
    criterion = HomoscedasticMTLLoss().to(TrainConfig.DEVICE)
    optimizer = AdamW([{"params": model.parameters()}, {"params": criterion.log_vars}], lr=TrainConfig.LEARNING_RATE, weight_decay=TrainConfig.WEIGHT_DECAY)
    scheduler = CosineAnnealingLR(optimizer, T_max=TrainConfig.NUM_EPOCHS, eta_min=1e-6)

    best_val_loss, patience_counter, best_epoch = float("inf"), 0, 0
    safe_name = fold_name.replace("/", "_").replace(" ", "_")
    ckpt_path = os.path.join(output_dir, f"{safe_name}_best.pt")

    for epoch in range(TrainConfig.NUM_EPOCHS):
        train_m = train_epoch(model, train_loader, optimizer, criterion, TrainConfig.DEVICE)
        val_m = evaluate(model, val_loader, criterion, TrainConfig.DEVICE, "Val")
        scheduler.step()
        ts, vs = train_m.get_summary(), val_m.get_summary()
        lr_now = optimizer.param_groups[0]["lr"]
        logger.log_epoch(fold_name, strategy, epoch + 1, "train", ts["loss_total"], ts["loss_cls"], ts["loss_reg"], ts["hazard_accuracy"], ts["hazard_f1_macro"], ts["severity_rmse"], ts["severity_r2"], lr=lr_now)
        logger.log_epoch(fold_name, strategy, epoch + 1, "val", vs["loss_total"], vs["loss_cls"], vs["loss_reg"], vs["hazard_accuracy"], vs["hazard_f1_macro"], vs["severity_rmse"], vs["severity_r2"], lr=lr_now)

        if vs["loss_total"] < best_val_loss:
            best_val_loss, patience_counter, best_epoch = vs["loss_total"], 0, epoch + 1
            torch.save(model.state_dict(), ckpt_path)
        else:
            patience_counter += 1
            if patience_counter >= TrainConfig.PATIENCE:
                print(f"  Early stopping at epoch {epoch+1} (best: {best_epoch})"); break
        if (epoch + 1) % 10 == 0 or epoch == 0:
            print(f"  Epoch {epoch+1:2d}/{TrainConfig.NUM_EPOCHS} | Train: {ts['loss_total']:.4f} Acc:{ts['hazard_accuracy']:.3f} mF1:{ts['hazard_f1_macro']:.3f} | Val: {vs['loss_total']:.4f} Acc:{vs['hazard_accuracy']:.3f} mF1:{vs['hazard_f1_macro']:.3f}")

    model.load_state_dict(torch.load(ckpt_path))
    test_metrics = evaluate(model, test_loader, criterion, TrainConfig.DEVICE, "Test")
    s = test_metrics.get_summary()
    print(f"  TEST Acc={s['hazard_accuracy']:.4f} F1w={s['hazard_f1']:.4f} F1macro={s['hazard_f1_macro']:.4f} RMSE={s['severity_rmse']:.4f} R2={s['severity_r2']:.4f}")

    logger.log_fold_result(strategy, fold_name, dict(s, n_test=len(test_loader.dataset)))
    per_class = test_metrics.get_per_class_metrics()
    logger.log_per_class(strategy, fold_name, per_class)
    tier_df = test_metrics.get_tier_verification()
    logger.log_tier_verification(strategy, fold_name, tier_df)
    logger.log_raw_predictions(strategy, fold_name, test_metrics.get_raw_predictions_dict())

    stat_val = StatisticalValidator()
    acc_ci = stat_val.bootstrap_ci(test_metrics.hazard_targets, test_metrics.hazard_preds, lambda yt, yp: accuracy_score(yt, yp))
    logger.log_statistical("bootstrap_accuracy", strategy, dict(fold=fold_name, **acc_ci))
    f1_ci = stat_val.bootstrap_ci(test_metrics.hazard_targets, test_metrics.hazard_preds, lambda yt, yp: f1_score(yt, yp, average="macro", zero_division=0))
    logger.log_statistical("bootstrap_f1_macro", strategy, dict(fold=fold_name, **f1_ci))

    if test_metrics.hazard_logits_all:
        all_logits = np.concatenate(test_metrics.hazard_logits_all, axis=0)
        all_labels = np.array(test_metrics.hazard_targets)
        probs_raw = torch.softmax(torch.from_numpy(all_logits), dim=-1).numpy()
        pred_class = probs_raw.argmax(axis=1)
        ece_before = VerificationMetrics.ece(probs_raw.max(axis=1), (pred_class == all_labels).astype(float))
        cal = Calibrator(num_classes).fit(all_logits, all_labels)
        probs_cal = cal.transform(all_logits)
        pred_cal = probs_cal.argmax(axis=1)
        ece_after = VerificationMetrics.ece(probs_cal.max(axis=1), (pred_cal == all_labels).astype(float))
        n_bins = 10; bin_edges = np.linspace(0, 1, n_bins + 1); mean_pred, frac_pos = [], []
        for i in range(n_bins):
            mask = (probs_cal.max(axis=1) >= bin_edges[i]) & (probs_cal.max(axis=1) < bin_edges[i + 1])
            if mask.sum() > 0:
                mean_pred.append(probs_cal.max(axis=1)[mask].mean())
                frac_pos.append((pred_cal[mask] == all_labels[mask]).mean())
        reliability = dict(mean_predicted=mean_pred, fraction_positive=frac_pos, ece=ece_after)
        logger.log_calibration(strategy, fold_name, ece_before, ece_after, reliability)
        viz.plot_reliability_diagram(reliability, f"{strategy}_{fold_name}", f"reliability_{safe_name}")

    viz.plot_confusion_matrix(test_metrics.get_confusion_matrix_normalized(), f"Confusion Matrix: {fold_name}", f"cm_{safe_name}")
    viz.plot_severity_scatter(test_metrics.severity_targets, test_metrics.severity_preds, s["severity_r2"], f"Severity: {fold_name}", f"severity_{safe_name}")
    viz.plot_per_class_radar(per_class, f"{strategy}_{fold_name}", f"radar_{safe_name}")

    epoch_df = pd.DataFrame(logger._epoch_logs)
    if not epoch_df.empty:
        fold_epochs = epoch_df[(epoch_df["fold"] == fold_name) & (epoch_df["strategy"] == strategy)]
        if not fold_epochs.empty:
            viz.plot_training_curves(fold_epochs, f"{strategy}_{fold_name}", f"training_{safe_name}")

    return dict(fold=fold_name, strategy=strategy, **s, n_test=len(test_loader.dataset), ckpt=ckpt_path)

# ============================================================================
# STRATEGY RUNNERS & MAIN
# ============================================================================
def _run_dirs(base, num_classes, output_dir, prefix, logger, viz, strategy_key, chain=False):
    results, prev_ckpt = [], None
    if not os.path.isdir(base):
        print(f"  WARNING: {base} not found - skipping"); return []
    for fd in sorted(glob.glob(os.path.join(base, "*"))):
        if not os.path.isdir(fd): continue
        fn = os.path.basename(fd)
        r = train_single_fold(f"{prefix}{fn}", os.path.join(fd, "train_events.csv"), os.path.join(fd, "val_events.csv"), os.path.join(fd, "test_events.csv"),
                              num_classes, output_dir, logger, viz, strategy_key, init_from=prev_ckpt if chain else None)
        if chain: prev_ckpt = r["ckpt"]
        results.append(r)
    return results

STRATEGY_MAP_KEYS = ["event_kfold", "spatial_lodo", "temporal", "spatio_temporal", "grouped_kfold", "rolling_origin"]

STRATEGY = "grouped_kfold"


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--strategy", default=STRATEGY,
                        choices=STRATEGY_MAP_KEYS + ["all"],
                        help="Single strategy name")
    parser.add_argument("--strategies", default=None,
                        help="Comma-separated list, e.g. grouped_kfold,rolling_origin")

    in_notebook = 'ipykernel' in sys.modules or 'IPython' in sys.modules
    if in_notebook:
        args = parser.parse_args(args=[])
    else:
        args = parser.parse_args()

    # Resolve which strategies to run
    if args.strategies:
        selected = [s.strip() for s in args.strategies.split(",")]
    elif isinstance(STRATEGY, list):
        selected = STRATEGY
    elif STRATEGY == "all":
        selected = STRATEGY_MAP_KEYS
    else:
        selected = [STRATEGY]

    # Validate selections
    invalid = [s for s in selected if s not in STRATEGY_MAP_KEYS]
    if invalid:
        print(f"ERROR: Unknown strategies: {invalid}")
        print(f"Valid options: {STRATEGY_MAP_KEYS + ['all']}")
        return

    print("=" * 80)
    print("HAZARDNET TRAINING PIPELINE")
    print(f"Running strategies: {selected}")
    print("=" * 80)

    logger = ExperimentLogger(TrainConfig.OUTPUT_DIR)
    viz = PublicationVisualizer(logger.dirs["figures"])

    with open(TrainConfig.CONFIG_PATH) as f:
        config = json.load(f)
    num_classes = config["n_classes"]
    print(f"Classes ({num_classes}): {config['hazard_types']}")
    print(f"Master HDF5: {TrainConfig.MASTER_H5_PATH}")
    print(f"Device: {TrainConfig.DEVICE}")

    strategy_defs = {
        "event_kfold": ("Event-Based 5-Fold CV", lambda n, o: _run_dirs(os.path.join(TrainConfig.EXPERIMENTAL_DIR, "event_kfold"), n, o, "event_kfold_", logger, viz, "event_kfold")),
        "spatial_lodo": ("Spatial LODO", lambda n, o: _run_dirs(os.path.join(TrainConfig.EXPERIMENTAL_DIR, "spatial_lodo"), n, o, "", logger, viz, "spatial_lodo")),
        "temporal": ("Temporal Split", lambda n, o: _run_dirs(os.path.join(TrainConfig.EXPERIMENTAL_DIR, "temporal_split"), n, o, "temporal_", logger, viz, "temporal")),
        "spatio_temporal": ("Spatio-Temporal", lambda n, o: _run_dirs(os.path.join(TrainConfig.EXPERIMENTAL_DIR, "spatio_temporal"), n, o, "", logger, viz, "spatio_temporal")),
        "grouped_kfold": ("Grouped K-Fold [LEAKAGE-SAFE]", lambda n, o: _run_dirs(os.path.join(TrainConfig.EXPERIMENTAL_DIR, "grouped_kfold"), n, o, "grouped_", logger, viz, "grouped_kfold")),
        "rolling_origin": ("Rolling-Origin [DEPLOYMENT GATE]", lambda n, o: _run_dirs(os.path.join(TrainConfig.EXPERIMENTAL_DIR, "rolling_origin"), n, o, "rolling_", logger, viz, "rolling_origin", chain=True)),
    }

    # Build execution list in canonical order
    strategies = [(k, strategy_defs[k]) for k in STRATEGY_MAP_KEYS if k in selected]
    all_results = {}

    for key, (name, fn) in strategies:
        print(f"\n{'='*80}\nSTRATEGY: {name}\n{'='*80}")
        out = os.path.join(TrainConfig.OUTPUT_DIR, key)
        os.makedirs(out, exist_ok=True)
        results = fn(num_classes, out)
        all_results[key] = results
        if results:
            for metric in ("hazard_accuracy", "hazard_f1", "hazard_f1_macro",
                           "severity_rmse", "severity_r2"):
                vals = [r.get(metric, r.get("accuracy", 0)) for r in results]
                print(f"  {metric}: {np.mean(vals):.4f} +/- {np.std(vals):.4f}")

    # Cross-strategy comparison table
    print(f"\n{'='*80}\nCROSS-STRATEGY COMPARISON\n{'='*80}")
    comparison_rows = []
    for key, (name, _) in strategies:
        results = all_results.get(key, [])
        if results:
            accs = [r.get("hazard_accuracy", r.get("accuracy", 0)) for r in results]
            f1s = [r.get("hazard_f1_macro", r.get("f1_macro", 0)) for r in results]
            rmses = [r.get("severity_rmse", r.get("rmse", 0)) for r in results]
            r2s = [r.get("severity_r2", r.get("r2", 0)) for r in results]
            comparison_rows.append({
                "Strategy": name,
                "N_Folds": len(results),
                "Accuracy_mean": np.mean(accs), "Accuracy_std": np.std(accs),
                "F1_macro_mean": np.mean(f1s), "F1_macro_std": np.std(f1s),
                "RMSE_mean": np.mean(rmses), "RMSE_std": np.std(rmses),
                "R2_mean": np.mean(r2s), "R2_std": np.std(r2s),
                "Total_Test_Events": sum(r.get("n_test", 0) for r in results),
            })

    if comparison_rows:
        df_comp = pd.DataFrame(comparison_rows)
        print(df_comp.to_string(index=False))
        logger.save_cross_strategy(df_comp)
        viz.plot_cross_strategy_comparison(df_comp, "cross_strategy_comparison")

    # Statistical comparison (if both leaky and safe baselines were run)
    if "grouped_kfold" in all_results and "event_kfold" in all_results:
        gk = all_results["grouped_kfold"]
        ek = all_results["event_kfold"]
        if gk and ek:
            gk_accs = [r.get("hazard_accuracy", 0) for r in gk]
            ek_accs = [r.get("hazard_accuracy", 0) for r in ek]
            cohens_d = StatisticalValidator.cohens_d(ek_accs, gk_accs)
            print(f"\n  Cohen's d (event_kfold vs grouped_kfold): {cohens_d:.4f}")
            print(f"  Interpretation: {'large' if abs(cohens_d)>0.8 else 'medium' if abs(cohens_d)>0.5 else 'small'} effect")
            logger.log_statistical("cohens_d_leakage_vs_safe", "comparison", dict(effect_size=cohens_d))

    # Deployment gate
    ro = all_results.get("rolling_origin", [])
    if ro:
        mF1 = np.mean([r.get("hazard_f1_macro", r.get("f1_macro", 0)) for r in ro])
        gate_pass = mF1 >= 0.5
        gate_data = dict(strategy="rolling_origin", macro_f1=float(mF1), threshold=0.5, gate_pass=bool(gate_pass),
                         recommendation="ADVISORY BETA DEPLOYMENT PERMITTED" if gate_pass else "NO-GO FOR PUBLIC ALERTING",
                         n_folds=len(ro), timestamp=datetime.now().isoformat())
        logger.save_deployment_gate(gate_data)
        print(f"\nDEPLOYMENT GATE (Rolling-Origin Macro-F1={mF1:.3f}, need >=0.5): "
              f"{' PASS - Advisory Beta' if gate_pass else 'NO-GO'}")

    # Leakage Audit Waterfall
    audit_data = {
        "event_kfold_acc": np.mean([r.get("hazard_accuracy", 0) for r in all_results.get("event_kfold", [])]) if all_results.get("event_kfold") else 0.0,
        "spatial_lodo_acc": np.mean([r.get("hazard_accuracy", 0) for r in all_results.get("spatial_lodo", [])]) if all_results.get("spatial_lodo") else 0.0,
        "grouped_kfold_acc": np.mean([r.get("hazard_accuracy", 0) for r in all_results.get("grouped_kfold", [])]) if all_results.get("grouped_kfold") else 0.0,
        "rolling_origin_acc": np.mean([r.get("hazard_accuracy", 0) for r in all_results.get("rolling_origin", [])]) if all_results.get("rolling_origin") else 0.0,
    }
    viz.plot_leakage_waterfall(audit_data, "leakage_degradation_waterfall")
    logger.log_leakage_audit(audit_data)

    # Save all artifacts
    logger.save_all()
    print(f"\n ALL EXPERIMENTS COMPLETE. Results saved to: {logger.run_dir}")

if __name__ == "__main__":
    main()


HAZARDNET TRAINING PIPELINE
Running strategies: ['grouped_kfold']
  Experiment Logger initialized: /content/drive/MyDrive/HazardNet_Deployment/HazardNet_Deployment_Bundles/Results/HazardNet_Model_Training_Results/HazardNet/run_20260919_104831
Classes (8): ['Cold Wave', 'Drought', 'Fire', 'Flash Flood', 'Flood', 'Heat Wave', 'Severe Local Storm', 'Tropical Cyclone']
Master HDF5: /content/drive/MyDrive/HazardNet_Deployment/tensors_output/HazardNet_Event_Based_Datasets/master_tensors.h5
Device: cuda

STRATEGY: Grouped K-Fold [LEAKAGE-SAFE]

FOLD: grouped_fold_0
  Train: 1904, Val: 409, Test: 618


Val: 100%|██████████| 26/26 [00:06<00:00,  4.07batch/s]


  Epoch  1/50 | Train: 1.1445 Acc:0.476 mF1:0.308 | Val: 0.6235 Acc:0.677 mF1:0.551


Val: 100%|██████████| 26/26 [00:05<00:00,  4.54batch/s]


  Epoch 10/50 | Train: -1.6173 Acc:0.955 mF1:0.950 | Val: -1.9257 Acc:0.980 mF1:0.981


...

  Epoch 40/50 | Train: -4.7284 Acc:0.991 mF1:0.992 | Val: -4.8092 Acc:0.993 mF1:0.985


Val: 100%|██████████| 26/26 [00:05<00:00,  4.38batch/s]


  Epoch 50/50 | Train: -4.9849 Acc:0.995 mF1:0.995 | Val: -4.9395 Acc:0.995 mF1:0.987


Test: 100%|██████████| 36/36 [00:09<00:00,  3.73batch/s]


  TEST Acc=0.9965 F1w=0.9965 F1macro=0.9979 RMSE=0.1288 R2=0.8725
  hazard_accuracy: 0.9866 +/- 0.0102
  hazard_f1: 0.9866 +/- 0.0103
  hazard_f1_macro: 0.9872 +/- 0.0111
  severity_rmse: 0.1280 +/- 0.0125
  severity_r2: 0.8744 +/- 0.0213

CROSS-STRATEGY COMPARISON
                     Strategy  N_Folds  Accuracy_mean  Accuracy_std  F1_macro_mean  F1_macro_std  RMSE_mean  RMSE_std  R2_mean   R2_std  Total_Test_Events
Grouped K-Fold [LEAKAGE-SAFE]        5       0.986592      0.010217       0.987187      0.011085   0.128035   0.01252 0.874396 0.021291               2931
  Saved cross-strategy comparison

SAVING ALL EXPERIMENT ARTIFACTS -> /content/drive/MyDrive/HazardNet_Deployment/HazardNet_Deployment_Bundles/Results/HazardNet_Model_Training_Results/HazardNet/run_20260919_104831
  Saved 486 epoch log entries -> /content/drive/MyDrive/HazardNet_Deployment/HazardNet_Deployment_Bundles/Results/HazardNet_Model_Training_Results/HazardNet/run_20260919_104831/logs/training_logs.csv
  Saved 5 

In [ ]:
BEST_PT = '/content/drive/MyDrive/HazardNet_Deployment/data/HazardNet_event_based_model_outputs/event_kfold/event_kfold_fold2_best.pt'

assert os.path.exists(BEST_PT), f"Training did not produce {BEST_PT}"
print(f"Trained model saved to {BEST_PT} ({os.path.getsize(BEST_PT)/1e6:.2f} MB)")

Trained model saved to /content/drive/MyDrive/HazardNet_Deployment/data/HazardNet_event_based_model_outputs/event_kfold/event_kfold_fold2_best.pt (0.57 MB)


In [ ]:
import os, json, shutil
os.makedirs(BUNDLE_DIR, exist_ok=True)

In [ ]:
import os
import json
import glob
import subprocess
import shutil
import sys
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import h5py
from tqdm import tqdm


class DeployConfig:
    BEST_CHECKPOINT = '/content/drive/MyDrive/HazardNet_Deployment/data/HazardNet_event_based_model_outputs/event_kfold/event_kfold_fold2_best.pt'
    MASTER_H5_PATH = '/content/drive/MyDrive/HazardNet_Deployment/tensors_output/HazardNet_Event_Based_Datasets/master_tensors.h5'
    CONFIG_PATH = '/content/drive/MyDrive/HazardNet_Deployment/tensors_output/HazardNet_Event_Based_Datasets/dataset_config.json'
    NORMALIZATION_STATS_PATH = '/content/drive/MyDrive/HazardNet_Deployment/tensors_output/normalization_stats.json'
    OUTPUT_DIR = '/content/drive/MyDrive/HazardNet_Deployment/HazardNet_Deployment_Bundles'

    IN_CHANNELS = 15
    NUM_HAZARDS = 8
    INPUT_SHAPE = (1, 15, 10, 64, 64)
    NUM_REPRESENTATIVE_SAMPLES = 100

    BAND_NAMES = [
        'SAR_VV', 'SAR_VH', 'Blue', 'Red', 'NIR', 'SWIR',
        'Temp_2m', 'Precip', 'Max_Temp', 'Min_Temp',
        'Soil_W1', 'Soil_W3', 'Soil_T1', 'Dewpoint', 'Solar_Rad'
    ]

os.makedirs(DeployConfig.OUTPUT_DIR, exist_ok=True)


class NormalizationStats:
    """Robust Normalization Stats Loader that adapts to various JSON schemas."""
    def __init__(self, stats_path: str, band_names: list):
        print(f"Loading normalization stats from: {stats_path}")
        if not os.path.exists(stats_path):
            raise FileNotFoundError(f"Normalization stats file not found: {stats_path}")

        with open(stats_path, 'r') as f:
            self.stats = json.load(f)

        self.band_names = band_names
        self.means_list = []
        self.stds_list = []

        self._parse_and_validate()

        self.means = np.array(self.means_list, dtype=np.float32).reshape(1, -1, 1, 1, 1)
        self.stds = np.array(self.stds_list, dtype=np.float32).reshape(1, -1, 1, 1, 1)
        self.stds = np.where(self.stds < 1e-8, 1.0, self.stds)

        print(f"  [OK] Successfully loaded normalization stats for {len(self.band_names)} bands")

    def _parse_and_validate(self):
        data = self.stats

        if isinstance(data, dict):
            for nested_key in ['per_band', 'bands', 'stats', 'band_stats']:
                if nested_key in data and isinstance(data[nested_key], (dict, list)):
                    data = data[nested_key]
                    break

        if isinstance(data, dict) and any(k in data for k in ['mean', 'means']) and any(k in data for k in ['std', 'stds']):
            m_key = 'mean' if 'mean' in data else 'means'
            s_key = 'std' if 'std' in data else 'stds'
            means_data, stds_data = data[m_key], data[s_key]

            if isinstance(means_data, list) and isinstance(stds_data, list):
                if len(means_data) == len(self.band_names):
                    self.means_list = [float(m) for m in means_data]
                    self.stds_list = [float(s) for s in stds_data]
                    return
                else:
                    raise ValueError(f"Stats list length ({len(means_data)}) != expected bands ({len(self.band_names)})")

            elif isinstance(means_data, dict) and isinstance(stds_data, dict):
                means_lower = {str(k).lower(): v for k, v in means_data.items()}
                stds_lower = {str(k).lower(): v for k, v in stds_data.items()}
                for i, b in enumerate(self.band_names):
                    b_lower, b_idx = b.lower(), str(i)
                    if b_lower in means_lower and b_lower in stds_lower:
                        self.means_list.append(float(means_lower[b_lower]))
                        self.stds_list.append(float(stds_lower[b_lower]))
                    elif b_idx in means_lower and b_idx in stds_lower:
                        self.means_list.append(float(means_lower[b_idx]))
                        self.stds_list.append(float(stds_lower[b_idx]))
                    else:
                        raise ValueError(f"Could not find mean/std for band '{b}' in stats dict")
                return

        if isinstance(data, dict):
            key_map = {str(k).lower(): v for k, v in data.items() if isinstance(v, dict)}
            missing = []
            for i, band in enumerate(self.band_names):
                band_lower, band_idx = band.lower(), str(i)
                target = key_map.get(band_lower) or key_map.get(band_idx)
                if target is not None:
                    m_val = target.get('mean', target.get('means'))
                    s_val = target.get('std', target.get('stds'))
                    if m_val is not None and s_val is not None:
                        self.means_list.append(float(m_val))
                        self.stds_list.append(float(s_val))
                        continue
                missing.append(band)
            if not missing:
                return
            raise ValueError(f"Normalization stats missing bands: {missing}.")

        if isinstance(data, list) and all(isinstance(x, dict) for x in data):
            band_map = {}
            for entry in data:
                b_name = entry.get('band', entry.get('name', entry.get('band_name')))
                if b_name is not None:
                    band_map[str(b_name).lower()] = entry
            for i, band in enumerate(self.band_names):
                entry = band_map.get(band.lower()) or band_map.get(str(i))
                if entry and 'mean' in entry and 'std' in entry:
                    self.means_list.append(float(entry['mean']))
                    self.stds_list.append(float(entry['std']))
                else:
                    raise ValueError(f"Missing stats entry for band '{band}' in list of stats.")
            return

        raise ValueError("Unrecognized normalization stats JSON structure.")

    def normalize(self, tensor: np.ndarray) -> np.ndarray:
        if tensor.ndim == 4:
            means, stds = self.means[0], self.stds[0]
        elif tensor.ndim == 5:
            means, stds = self.means, self.stds
        else:
            raise ValueError(f"Expected 4D or 5D tensor, got {tensor.ndim}D")
        return (tensor.astype(np.float32) - means) / stds

    def to_dict(self) -> dict:
        return {
            'means': {b: float(self.means_list[i]) for i, b in enumerate(self.band_names)},
            'stds': {b: float(self.stds_list[i]) for i, b in enumerate(self.band_names)},
            'band_order': self.band_names,
            'normalization_type': 'z_score',
        }


class DepthwiseSeparableConv3d(nn.Module):
    def __init__(self, in_channels, out_channels, kernel_size=3, padding=1):
        super().__init__()
        self.depthwise = nn.Conv3d(in_channels, in_channels, kernel_size, padding=padding, groups=in_channels, bias=False)
        self.pointwise = nn.Conv3d(in_channels, out_channels, kernel_size=1, bias=False)
        self.bn = nn.BatchNorm3d(out_channels)

    def forward(self, x):
        return self.bn(self.pointwise(self.depthwise(x)))


class SEBlock3D(nn.Module):
    def __init__(self, channels, reduction=4):
        super().__init__()
        self.fc = nn.Sequential(
            nn.AdaptiveAvgPool3d(1), nn.Flatten(),
            nn.Linear(channels, channels // reduction, bias=False),
            nn.ReLU(inplace=True),
            nn.Linear(channels // reduction, channels, bias=False),
            nn.Sigmoid())

    def forward(self, x):
        w = self.fc(x).unsqueeze(-1).unsqueeze(-1).unsqueeze(-1)
        return x * w


class HazardNetCNN(nn.Module):
    def __init__(self, in_channels=15, num_hazards=8):
        super().__init__()
        self.block1 = nn.Sequential(DepthwiseSeparableConv3d(in_channels, 32), nn.ReLU(True), SEBlock3D(32), nn.MaxPool3d((1, 2, 2)))
        self.block2 = nn.Sequential(DepthwiseSeparableConv3d(32, 64), nn.ReLU(True), SEBlock3D(64), nn.MaxPool3d((2, 2, 2)))
        self.block3 = nn.Sequential(DepthwiseSeparableConv3d(64, 128), nn.ReLU(True), SEBlock3D(128), nn.MaxPool3d((1, 2, 2)))
        self.block4 = nn.Sequential(DepthwiseSeparableConv3d(128, 256), nn.ReLU(True), SEBlock3D(256), nn.MaxPool3d((1, 2, 2)))
        self.global_pool = nn.AdaptiveAvgPool3d(1)
        self.shared_fc = nn.Sequential(nn.Linear(256, 128), nn.ReLU(True), nn.Dropout(0.3))
        self.hazard_head = nn.Linear(128, num_hazards)
        self.severity_head = nn.Sequential(nn.Linear(128, 64), nn.ReLU(True), nn.Linear(64, 1), nn.Sigmoid())

    def forward(self, x):
        x = self.block4(self.block3(self.block2(self.block1(x))))
        x = self.global_pool(x).view(x.size(0), -1)
        x = self.shared_fc(x)
        return self.hazard_head(x), self.severity_head(x).squeeze(1)


def load_model_and_stats():
    print("=" * 70)
    print("STEP 1: Loading Model & Pre-computed Normalization Stats")
    print("=" * 70)
    norm_stats = NormalizationStats(DeployConfig.NORMALIZATION_STATS_PATH, DeployConfig.BAND_NAMES)

    model = HazardNetCNN(DeployConfig.IN_CHANNELS, DeployConfig.NUM_HAZARDS)
    state_dict = torch.load(DeployConfig.BEST_CHECKPOINT, map_location='cpu')
    model.load_state_dict(state_dict)
    model.eval()

    params = sum(p.numel() for p in model.parameters())
    print(f"  [OK] Model loaded: {params:,} params (~{params * 4 / 1024**2:.2f} MB FP32)")
    return model, norm_stats


def export_to_onnx(model):
    print("\n" + "=" * 70)
    print("STEP 2: Exporting to ONNX")
    print("=" * 70)
    output_path = os.path.join(DeployConfig.OUTPUT_DIR, 'hazardnet.onnx')
    dummy_input = torch.randn(*DeployConfig.INPUT_SHAPE)

    print("  Tracing model with torch.jit.trace to bypass onnxscript registry bugs...")
    try:
        export_target = torch.jit.trace(model, dummy_input)
    except Exception as e:
        print(f"  [WARN] Tracing warning ({e}), falling back to PyTorch model")
        export_target = model

    torch.onnx.export(
        export_target, dummy_input, output_path,
        export_params=True, opset_version=17, do_constant_folding=True,
        input_names=['input'],
        output_names=['hazard_logits', 'severity_pred'],
        dynamic_axes={'input': {0: 'batch_size'}, 'hazard_logits': {0: 'batch_size'}, 'severity_pred': {0: 'batch_size'}},
        dynamo=False
    )

    import onnx
    onnx_model = onnx.load(output_path)
    onnx.checker.check_model(onnx_model)
    print(f"  [OK] ONNX exported: {output_path} ({os.path.getsize(output_path) / 1024**2:.2f} MB)")
    return output_path


def create_representative_dataset(norm_stats: NormalizationStats):
    print("\n" + "=" * 70)
    print(f"STEP 3: Creating Representative Dataset ({DeployConfig.NUM_REPRESENTATIVE_SAMPLES} samples)")
    print("=" * 70)

    event_kfold_dir = os.path.dirname(DeployConfig.MASTER_H5_PATH)
    train_csvs = sorted(glob.glob(os.path.join(event_kfold_dir, 'event_kfold/fold_*/train_events.csv')))
    if not train_csvs:
        raise FileNotFoundError(f"No train CSVs found in {event_kfold_dir}")

    samples_per_fold = max(1, DeployConfig.NUM_REPRESENTATIVE_SAMPLES // len(train_csvs))
    all_event_ids = []
    for csv_path in train_csvs:
        fold_df = pd.read_csv(csv_path)
        fold_sample = fold_df.sample(n=min(samples_per_fold, len(fold_df)), random_state=42)
        all_event_ids.extend(fold_sample['event_id'].astype(str).tolist())
    all_event_ids = all_event_ids[:DeployConfig.NUM_REPRESENTATIVE_SAMPLES]

    h5f = h5py.File(DeployConfig.MASTER_H5_PATH, 'r')
    samples = []
    for eid in tqdm(all_event_ids, desc="Loading & normalizing"):
        raw_tensor = h5f[f'tensors/{eid}'][:]
        normalized = norm_stats.normalize(raw_tensor)
        samples.append(normalized[np.newaxis, ...])
    h5f.close()

    print(f"  [OK] Created {len(samples)} normalized representative samples")
    return samples


def convert_to_tflite(onnx_path, representative_samples):
    print("\n" + "=" * 70)
    print("STEP 4: Converting to TFLite")
    print("=" * 70)

    # Install onnx2tf if not already installed
    try:
        import onnx2tf
    except ImportError:
        print("  Installing onnx2tf...")
        subprocess.run([sys.executable, '-m', 'pip', 'install', 'onnx2tf'], check=True, capture_output=True, text=True)
        print("  onnx2tf installed successfully.")

    import tensorflow as tf

    tflite_dir = os.path.join(DeployConfig.OUTPUT_DIR, 'tflite')
    os.makedirs(tflite_dir, exist_ok=True)
    tf_saved_model_dir = os.path.join(tflite_dir, 'tf_saved_model')

    print("  Converting ONNX -> TF SavedModel / TFLite...")

    # Robust CLI execution using sys.executable to avoid PATH issues in Kaggle/Colab
    cmd = [sys.executable, '-m', 'onnx2tf', '-i', onnx_path, '-o', tf_saved_model_dir, '-osd', '-nuo']
    print(f"  Running command: {' '.join(cmd)}")
    result = subprocess.run(cmd, capture_output=True, text=True)

    # Print stdout and stderr for debugging
    print("onnx2tf stdout:")
    print(result.stdout)
    print("onnx2tf stderr:")
    print(result.stderr)

    # Raise an exception if onnx2tf failed
    result.check_returncode()
    print("  [OK] onnx2tf command executed successfully.")

    def _find_saved_model(path):
        for root, _, files in os.walk(path):
            if 'saved_model.pb' in files: return root
        return None

    saved_pb_path = _find_saved_model(tf_saved_model_dir)
    fp32_model_bytes = None

    if saved_pb_path:
        print(f"  [OK] Found TF SavedModel at: {saved_pb_path}")
        print("  Converting SavedModel to Pure FP32 TFLite...")
        converter = tf.lite.TFLiteConverter.from_saved_model(saved_pb_path)
        converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS, tf.lite.OpsSet.SELECT_TF_OPS]
        converter.inference_input_type = tf.float32
        converter.inference_output_type = tf.float32
        fp32_model_bytes = converter.convert()
    else:
        # Fallback: onnx2tf often outputs .tflite directly if SavedModel generation fails
        direct_tflite_files = glob.glob(os.path.join(tf_saved_model_dir, "*float32.tflite"))
        if not direct_tflite_files:
            direct_tflite_files = glob.glob(os.path.join(tf_saved_model_dir, "*.tflite"))

        if direct_tflite_files:
            saved_pb_path = direct_tflite_files[0]
            print(f"  [OK] Found direct FP32 TFLite model: {saved_pb_path}")
            with open(saved_pb_path, 'rb') as f:
                fp32_model_bytes = f.read()
        else:
            raise RuntimeError(f"TF SavedModel or TFLite not created at '{tf_saved_model_dir}'.")

    fp32_path = os.path.join(tflite_dir, 'hazardnet_fp32.tflite')
    with open(fp32_path, 'wb') as f:
        f.write(fp32_model_bytes)

    print("\n  [INFO] ARCHITECTURE LIMITATION DETECTED")
    print("  TensorFlow Lite's native 'CONV_3D' kernel strictly requires FLOAT32 tensors.")
    print("  Applying Optimize.DEFAULT (INT8) causes a runtime crash in conv3d.cc.")
    print("  To guarantee edge compatibility, INT8 quantization is safely bypassed.")

    # Save FP32 as the final optimized model for edge deployment
    int8_path = os.path.join(tflite_dir, 'hazardnet_optimized_fp32.tflite')
    with open(int8_path, 'wb') as f:
        f.write(fp32_model_bytes)

    fp32_mb = len(fp32_model_bytes) / (1024 ** 2)
    print(f"\n  TFLite Results:")
    print(f"     Model Size: {fp32_mb:.2f} MB (Pure FP32)")
    print(f"     {'[OK] UNDER 150 MB TARGET' if fp32_mb < 150 else '[WARN] EXCEEDS 150 MB'}")

    return fp32_path, int8_path


def golden_parity_test(model, norm_stats, int8_path, n_samples=50):
    print("\n" + "=" * 70)
    print(f"STEP 5: Golden Parity Test ({n_samples} samples)")
    print("=" * 70)

    import tensorflow as tf

    event_kfold_dir = os.path.dirname(DeployConfig.MASTER_H5_PATH)
    test_csvs = sorted(glob.glob(os.path.join(event_kfold_dir, 'event_kfold/fold_*/test_events.csv')))
    if not test_csvs:
        print("  [WARN] No test CSVs found, skipping parity test")
        return 0.0, 0.0

    samples_per_fold = max(1, n_samples // len(test_csvs))
    all_event_ids = []
    for csv_path in test_csvs:
        fold_df = pd.read_csv(csv_path)
        fold_sample = fold_df.sample(n=min(samples_per_fold, len(fold_df)), random_state=42)
        all_event_ids.extend(fold_sample['event_id'].astype(str).tolist())
    all_event_ids = all_event_ids[:n_samples]

    h5f = h5py.File(DeployConfig.MASTER_H5_PATH, 'r')
    interpreter = tf.lite.Interpreter(model_path=int8_path)
    interpreter.allocate_tensors()
    inp_details = interpreter.get_input_details()[0]
    out_details = interpreter.get_output_details()

    tflite_input_shape = inp_details['shape']
    # Detect if TFLite expects NDHWC (1, 10, 64, 64, 15) instead of NCDHW (1, 15, 10, 64, 64)
    needs_transpose = (len(tflite_input_shape) == 5 and tflite_input_shape[1] != DeployConfig.IN_CHANNELS and tflite_input_shape[4] == DeployConfig.IN_CHANNELS)

    print(f"  [INFO] TFLite expected input shape: {tuple(tflite_input_shape)}")
    print(f"  [INFO] Transpose required (NCDHW -> NDHWC): {needs_transpose}")

    model.eval()
    hazard_agreements = 0
    severity_diffs = []

    for eid in tqdm(all_event_ids, desc="Parity test"):
        raw_tensor = h5f[f'tensors/{eid}'][:]
        normalized = norm_stats.normalize(raw_tensor)
        batch = normalized[np.newaxis, ...]

        with torch.no_grad():
            pt_hazard, pt_severity = model(torch.from_numpy(batch))
        pt_class = pt_hazard.argmax(dim=1).item()

        tflite_batch = np.transpose(batch, (0, 2, 3, 4, 1)) if needs_transpose else batch
        tflite_input = tflite_batch.astype(inp_details['dtype'])
        interpreter.set_tensor(inp_details['index'], tflite_input)
        interpreter.invoke()

        # Robust output extraction by shape rather than strict index
        tf_hazard, tf_severity = None, None
        for out in out_details:
            out_shape = out['shape']
            if len(out_shape) == 2 and out_shape[1] == DeployConfig.NUM_HAZARDS:
                tf_hazard = interpreter.get_tensor(out['index'])[0]
            elif len(out_shape) <= 2 and (out_shape[-1] == 1 or out_shape == (1,)):
                tf_severity = interpreter.get_tensor(out['index'])[0]
                if isinstance(tf_severity, np.ndarray):
                    tf_severity = tf_severity.item() if tf_severity.size == 1 else tf_severity[0]

        # Fallback if shape matching failed
        if tf_hazard is None or tf_severity is None:
            tf_hazard = interpreter.get_tensor(out_details[0]['index'])[0]
            tf_severity = interpreter.get_tensor(out_details[1]['index'])[0]
            if isinstance(tf_severity, np.ndarray):
                tf_severity = tf_severity.item() if tf_severity.size == 1 else tf_severity[0]

        if pt_class == int(np.argmax(tf_hazard)):
            hazard_agreements += 1
        severity_diffs.append(abs(pt_severity.item() - float(tf_severity)))

    h5f.close()
    agreement_pct = hazard_agreements / len(all_event_ids) * 100
    mean_sev_diff = np.mean(severity_diffs)

    print(f"\n  Parity Results:")
    print(f"     Hazard agreement: {agreement_pct:.1f}%")
    print(f"     Severity MAE (PT vs TFLite): {mean_sev_diff:.4f}")
    print(f"     Status: {'[OK] PASSED' if agreement_pct >= 95 else '[WARN] BELOW 95% THRESHOLD'}")
    return agreement_pct, mean_sev_diff


def create_deployment_bundle(norm_stats: NormalizationStats, int8_path, fp32_path):
    print("\n" + "=" * 70)
    print("STEP 6: Creating Deployment Bundle")
    print("=" * 70)

    bundle_dir = os.path.join(DeployConfig.OUTPUT_DIR, 'deployment_bundle')
    os.makedirs(bundle_dir, exist_ok=True)

    for src_path, dst_name in [(int8_path, 'hazardnet_int8.tflite'), (fp32_path, 'hazardnet_fp32.tflite')]:
        if src_path and os.path.exists(src_path):
            shutil.copy2(src_path, os.path.join(bundle_dir, dst_name))

    if os.path.exists(DeployConfig.CONFIG_PATH):
        with open(DeployConfig.CONFIG_PATH, 'r') as f:
            config = json.load(f)
        labels = {str(i): h for i, h in enumerate(config.get('hazard_types', []))}
    else:
        labels = {str(i): f"Hazard_{i}" for i in range(DeployConfig.NUM_HAZARDS)}

    with open(os.path.join(bundle_dir, 'labels.json'), 'w') as f:
        json.dump(labels, f, indent=2)

    preprocessing_config = {
        'normalization': norm_stats.to_dict(),
        'input_shape': list(DeployConfig.INPUT_SHAPE),
        'num_hazards': DeployConfig.NUM_HAZARDS,
        'outputs': {'hazard_logits': 'index_0', 'severity_pred': 'index_1'},
    }
    with open(os.path.join(bundle_dir, 'preprocessing_config.json'), 'w') as f:
        json.dump(preprocessing_config, f, indent=2)

    inference_script = '''#!/usr/bin/env python3
"""HazardNet Edge Inference with Pre-computed Normalization"""
import numpy as np, tensorflow as tf, json, time, sys

def load_normalization_stats(config_path='preprocessing_config.json'):
    with open(config_path) as f: config = json.load(f)
    norm = config['normalization']
    means = np.array([norm['means'][b] for b in norm['band_order']], dtype=np.float32).reshape(1, -1, 1, 1, 1)
    stds = np.array([norm['stds'][b] for b in norm['band_order']], dtype=np.float32).reshape(1, -1, 1, 1, 1)
    return means, np.where(stds < 1e-8, 1.0, stds)

def normalize(raw_tensor, means, stds):
    return (raw_tensor.astype(np.float32) - means) / stds

def predict(tflite_path, raw_tensor, means, stds, labels_path='labels.json'):
    normalized = normalize(raw_tensor, means, stds)

    # Transpose NCDHW -> NDHWC for TFLite if necessary
    if normalized.shape[1] == 15 and normalized.shape[2] == 10:
        normalized = np.transpose(normalized, (0, 2, 3, 4, 1))

    interp = tf.lite.Interpreter(model_path=tflite_path)
    interp.allocate_tensors()
    inp = interp.get_input_details()[0]
    outs = interp.get_output_details()

    start = time.perf_counter()
    interp.set_tensor(inp['index'], normalized.astype(inp['dtype']))
    interp.invoke()
    latency = (time.perf_counter() - start) * 1000

    # Robust output extraction by shape
    hazard, severity = None, None
    for out in outs:
        if len(out['shape']) == 2 and out['shape'][1] == 8:
            hazard = interp.get_tensor(out['index'])[0]
        elif len(out['shape']) <= 2:
            severity = interp.get_tensor(out['index'])[0]

    if hazard is None: hazard = interp.get_tensor(outs[0]['index'])[0]
    if severity is None: severity = interp.get_tensor(outs[1]['index'])[0]
    if isinstance(severity, np.ndarray):
        severity = severity.item() if severity.size == 1 else severity[0]

    with open(labels_path) as f: labels = json.load(f)
    cls = int(np.argmax(hazard))
    return {
        'hazard': labels[str(cls)],
        'confidence': float(np.max(hazard)),
        'severity': float(severity),
        'latency_ms': latency
    }

if __name__ == '__main__':
    model = sys.argv[1] if len(sys.argv) > 1 else 'hazardnet_int8.tflite'
    means, stds = load_normalization_stats()
    r = predict(model, np.random.randn(1, 15, 10, 64, 64).astype(np.float32), means, stds)
    print(f"Hazard: {r['hazard']} (conf: {r['confidence']:.3f}) | Severity: {r['severity']:.4f} | Latency: {r['latency_ms']:.1f} ms")
'''
    with open(os.path.join(bundle_dir, 'inference_example.py'), 'w') as f:
        f.write(inference_script)

    readme = f"""# HazardNet Edge Deployment Bundle\n\n## Contents\n- `hazardnet_int8.tflite` - Optimized FP32 model (TFLite CONV_3D requires FP32)\n- `hazardnet_fp32.tflite` - FP32 baseline model\n- `labels.json` - {DeployConfig.NUM_HAZARDS} hazard class labels\n- `preprocessing_config.json` - Pre-computed normalization stats + band order\n- `inference_example.py` - Standalone inference with normalization & NDHWC transpose\n\n## Input Spec\n- Shape: (1, 15, 10, 64, 64) - [batch, channels, timesteps, height, width]\n- Normalization: z-score with pre-computed per-band mean/std\n- Transpose: NCDHW -> NDHWC handled automatically by inference script\n"""
    with open(os.path.join(bundle_dir, 'README.md'), 'w') as f:
        f.write(readme)

    print(f"\n  [OK] Bundle created: {bundle_dir}")
    for item in sorted(os.listdir(bundle_dir)):
        fpath = os.path.join(bundle_dir, item)
        if os.path.isfile(fpath):
            print(f"     {item}: {os.path.getsize(fpath) / 1024:.1f} KB")
        else:
            print(f"     {item}/ (directory)")
    return bundle_dir


def main():
    print("=" * 70)
    print("HAZARDNET EDGE DEPLOYMENT CONVERTER")
    print("=" * 70)

    model, norm_stats = load_model_and_stats()
    onnx_path = export_to_onnx(model)
    rep_samples = create_representative_dataset(norm_stats)
    fp32_path, int8_path = convert_to_tflite(onnx_path, rep_samples)
    agreement, sev_diff = golden_parity_test(model, norm_stats, int8_path)
    bundle_dir = create_deployment_bundle(norm_stats, int8_path, fp32_path)

    print("\n" + "=" * 70)
    print("[OK] DEPLOYMENT CONVERSION COMPLETE")
    print("=" * 70)
    print(f"  Bundle: {bundle_dir}")
    print(f"  Parity: {agreement:.1f}% hazard agreement, {sev_diff:.4f} severity MAE")


if __name__ == '__main__':
    main()

HAZARDNET EDGE DEPLOYMENT CONVERTER
STEP 1: Loading Model & Pre-computed Normalization Stats
Loading normalization stats from: /content/drive/MyDrive/HazardNet_Deployment/tensors_output/normalization_stats.json
  [OK] Successfully loaded normalization stats for 15 bands
  [OK] Model loaded: 136,670 params (~0.52 MB FP32)

STEP 2: Exporting to ONNX
  Tracing model with torch.jit.trace to bypass onnxscript registry bugs...
  [OK] ONNX exported: /content/drive/MyDrive/HazardNet_Deployment/HazardNet_Deployment_Bundles/hazardnet.onnx (0.53 MB)

STEP 3: Creating Representative Dataset (100 samples)


Loading & normalizing: 100%|██████████| 100/100 [00:01<00:00, 53.77it/s]


  [OK] Created 100 normalized representative samples

STEP 4: Converting to TFLite
  Installing onnx2tf...
  onnx2tf installed successfully.
  Converting ONNX -> TF SavedModel / TFLite...
  Running command: /usr/bin/python3 -m onnx2tf -i /content/drive/MyDrive/HazardNet_Deployment/HazardNet_Deployment_Bundles/hazardnet.onnx -o /content/drive/MyDrive/HazardNet_Deployment/HazardNet_Deployment_Bundles/tflite/tf_saved_model -osd -nuo
onnx2tf stdout:

Automatic generation of each OP name started ========================================
Automatic generation of each OP name complete!

Model loaded ========================================================================

flatbuffer_direct fast path started ===============================================
flatbuffer_direct write timing: stage=float32 mode=builder_direct total=0.460s serialize=0.091s (sanitize=0.001s build=0.010s pack=0.080s output=0.000s) write=0.367s size=0.75MB
flatbuffer_direct write timing: stage=float16 mode=builder_direct 

Parity test: 100%|██████████| 50/50 [00:08<00:00,  5.72it/s]



  Parity Results:
     Hazard agreement: 100.0%
     Severity MAE (PT vs TFLite): 0.0000
     Status: [OK] PASSED

STEP 6: Creating Deployment Bundle

  [OK] Bundle created: /content/drive/MyDrive/HazardNet_Deployment/HazardNet_Deployment_Bundles/deployment_bundle
     README.md: 0.6 KB
     hazardnet_fp32.tflite: 772.0 KB
     hazardnet_int8.tflite: 772.0 KB
     inference_example.py: 2.5 KB
     labels.json: 0.2 KB
     preprocessing_config.json: 1.6 KB

[OK] DEPLOYMENT CONVERSION COMPLETE
  Bundle: /content/drive/MyDrive/HazardNet_Deployment/HazardNet_Deployment_Bundles/deployment_bundle
  Parity: 100.0% hazard agreement, 0.0000 severity MAE


In [ ]:
print("Conversion placeholder — wire in your actual ONNX/TF steps above.")
print("Bundle dir:", BUNDLE_DIR)

Conversion placeholder — wire in your actual ONNX/TF steps above.
Bundle dir: /content/drive/MyDrive/HazardNet_Deployment/HazardNet_Deployment_Bundles/deployment_bundle


In [ ]:
# Smoke-test the artifact exactly as Actions will: load with tflite-runtime
# (same 2 MB wheel used in the daily job) and run one inference.
import os
import numpy as np
import tensorflow as tf # Using tensorflow's tf.lite.Interpreter

TFLITE_PATH = '/content/drive/MyDrive/HazardNet_Deployment/HazardNet_Deployment_Bundles/deployment_bundle/hazardnet_fp32.tflite'

# Check if the TFLite file exists before attempting to load
if not os.path.exists(TFLITE_PATH):
    print(f"Error: TFLite model not found at {TFLITE_PATH}")
    print("Please ensure the previous TFLite conversion step completed successfully.")
else:
    interp = tf.lite.Interpreter(model_path=TFLITE_PATH)
    interp.allocate_tensors()
    inp = interp.get_input_details()[0]
    out = interp.get_output_details()
    dummy = np.random.randn(*inp['shape']).astype(inp['dtype'])
    interp.set_tensor(inp['index'], dummy)
    interp.invoke()
    for o in out:
        print(f"  output {o['name']}: shape={o['shape']} dtype={o['dtype']}")
    print(f"TFLite model OK  ({os.path.getsize(TFLITE_PATH)/1e6:.2f} MB)")

  output hazard_logits: shape=[1 8] dtype=<class 'numpy.float32'>
  output severity_pred: shape=[1] dtype=<class 'numpy.float32'>
TFLite model OK  (0.79 MB)


In [ ]:
!git config --global user.email "{GIT_USER_EMAIL}"
!git config --global user.name  "{GIT_USER_NAME}"

# Shallow-clone main so we don't pull the full history in Colab.
!rm -rf {REPO_DIR}
!git clone --depth 1 https://{GITHUB_PAT}@github.com/{GITHUB_USERNAME}/{GITHUB_REPO}.git {REPO_DIR}
%cd {REPO_DIR}

# Replace the Models/ directory contents with the freshly-converted bundle.
!rm -rf Models/*
!cp -r {BUNDLE_DIR}/* Models/
!ls -lh Models/

# Commit and push on a dedicated auto-ml branch, then open a PR via gh CLI
# (safer than force-pushing to main). If you prefer direct push, just
# `git checkout main && git push` instead.
import datetime as _dt
BRANCH = f"auto-ml/model-update-{_dt.datetime.utcnow().strftime('%Y%m%d-%H%M%S')}"
BRANCH_MSG = f"chore(model): auto-update TFLite from Colab training run ({_dt.date.today()})"
!git checkout -b {BRANCH}
!git add Models/
!git -c user.name="{GIT_USER_NAME}" -c user.email="{GIT_USER_EMAIL}" commit -m "{BRANCH_MSG}"
!git push -u origin {BRANCH}

# Optionally create a PR using the GitHub API (no extra deps).
import json, urllib.request
req = urllib.request.Request(
    f'https://api.github.com/repos/{GITHUB_USERNAME}/{GITHUB_REPO}/pulls',
    data=json.dumps({
        'title': BRANCH_MSG,
        'head': BRANCH,
        'base': 'main',
        'body': 'Automated TFLite model update from Colab training run.\n\n- [ ] Verify `daily_forecast.yml` smoke passes\n- [ ] Check model_version bump\n',
    }).encode(),
    headers={'Authorization': f'token {GITHUB_PAT}', 'Accept': 'application/vnd.github+json'},
)
try:
    resp = urllib.request.urlopen(req)
    pr = json.load(resp)
    print(f" PR opened: {pr['html_url']}")
except Exception as e:
    print(f"PR creation failed (push to branch succeeded): {e}")

Cloning into '/content/drive/MyDrive/HazardNet_Deployment/HazardNet'...
remote: Enumerating objects: 1316, done.
remote: Counting objects: 100% (1316/1316), done.
remote: Compressing objects: 100% (1151/1151), done.
remote: Total 1316 (delta 111), reused 1023 (delta 104), pack-reused 0 (from 0)
Receiving objects: 100% (1316/1316), 11.32 MiB | 9.29 MiB/s, done.
Resolving deltas: 100% (111/111), done.
Updating files: 100% (1145/1145), done.
/content/drive/MyDrive/HazardNet_Deployment/HazardNet
total 1.6M
-rw------- 1 root root 772K Sep 19 13:33 hazardnet_fp32.tflite
-rw------- 1 root root 772K Sep 19 13:33 hazardnet_int8.tflite
-rw------- 1 root root 2.5K Sep 19 13:33 inference_example.py
-rw------- 1 root root  169 Sep 19 13:33 labels.json
-rw------- 1 root root 1.6K Sep 19 13:33 preprocessing_config.json
-rw------- 1 root root  600 Sep 19 13:33 README.md
Switched to a new branch 'auto-ml/model-update-20260919-133305'
[auto-ml/model-update-20260919-133305 9c84673] chore(model): auto-upd